In [ ]:
pip install fastdtw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567858 sha256=08d75e8490d6ff679b23a1d259e2389f716d23f8942369b5cdb260268eb7472d
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train = pd.read_csv('/content/drive/MyDrive/국민대_무역_공모전/데이터/train.csv')
submission = pd.read_csv('/content/drive/MyDrive/국민대_무역_공모전/데이터/sample_submission.csv')

In [4]:
# ================================
# 1) 월별 집계
# ================================
monthly = (
    train.groupby(['item_id', 'year', 'month'])
         .agg({
             'value': 'sum',
             'weight': 'sum',
             'quantity': 'sum',
             'hs4': 'first'
         })
         .reset_index()
)

# ================================
# 2) year_month 생성
# ================================
monthly['year_month'] = pd.to_datetime(
    monthly['year'].astype(str) + '-' + monthly['month'].astype(str),
    format='%Y-%m'
)

# ================================
# 3) Value 피벗 (행: year_month, 열: item_id)
# ================================
pivot_value = (
    monthly.pivot_table(
        index='year_month',
        columns='item_id',
        values='value',
        fill_value=0
    )
    .sort_index()
)

# ================================
# 4) Weight 피벗 (행: year_month, 열: item_id)
# ================================
pivot_weight = (
    monthly.pivot_table(
        index='year_month',
        columns='item_id',
        values='weight',
        fill_value=0
    )
    .sort_index()
)

# ================================
# 5) Transpose (행 = item_id 로 변경)
# ================================
pivot = pivot_value.T           # (n_items × n_months)
pivot_weight_T = pivot_weight.T # 동일 구조

# ================================
# 6) 기본 출력
# ================================
print("✓ Value pivot:", pivot_value.shape)
print("✓ Weight pivot:", pivot_weight.shape)
pivot.head(5)


✓ Value pivot: (43, 100)
✓ Weight pivot: (43, 100)


year_month,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0


In [5]:
from scipy.stats import pearsonr

def safe_corr_with_pvalue(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0, 1.0
    corr, p_value = pearsonr(x, y)
    return float(corr), float(p_value) # 피어슨 상관계수, p-value

In [6]:
def calculate_score_advanced(corr, p_value, lag):
    """
    상관계수(70%) + p-value(20%) + lag(10%)
    """
    # 상관계수 (0~70점)
    corr_score = abs(corr) * 70

    # 통계적 유의성 (0~20점)
    if p_value < 0.001:
        sig_score = 20
    elif p_value < 0.01:
        sig_score = 15
    elif p_value < 0.05:
        sig_score = 10
    else:
        sig_score = 5

    # Lag (0~10점) - 짧을수록 좋음
    if lag <= 2:
        lag_score = 10
    elif lag <= 4:
        lag_score = 7
    elif lag <= 6:
        lag_score = 4
    else:
        lag_score = 1

    return corr_score + sig_score + lag_score

## DTW 값이 작을수록 패턴이 유사

DTW (Dynamic Time Warping)은 시계열 데이터 간의 유사성을 비교하기 위한 알고리즘이다. DTW는 시계열 데이터 간의 길이나 속도가 달라도, 이것을 고려하여 유사성을 측정할 수 있기 때문에 시계열 데이터 분석에 많이 활용된다.

왜 DTW를 쓰는가?
상관계수만으로는 놓치는 것:

- 속도 차이 (A는 천천히, B는 빠르게 변함)
- 타이밍 차이 (A가 먼저 변하고 B가 조금 늦게 따라감)
- 노이즈 (한두 달 튀는 값)

DTW는 이런 것까지 고려해서 "진짜 패턴 유사도"를 측정

In [7]:
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import StandardScaler

# DTW 함수
def dtw_distance(series_A, series_B):
    """
    Dynamic Time Warping 거리 계산
    """
    n, m = len(series_A), len(series_B)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(series_A[i-1] - series_B[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],
                dtw_matrix[i, j-1],
                dtw_matrix[i-1, j-1]
            )

    return dtw_matrix[n, m]

def calculate_normalized_dtw(x, y, lag):
    """
    정규화 + DTW 거리 계산
    x: 선행 품목
    y: 후행 품목
    lag: 시차
    """
    if len(x) <= lag or len(y) <= lag:
        return float('inf')

    x_aligned = x[:-lag]
    y_aligned = y[lag:]

    if len(x_aligned) < 5:
        return float('inf')

    # 정규화
    scaler = StandardScaler()
    normalized_x = scaler.fit_transform(x_aligned.reshape(-1, 1)).flatten()
    normalized_y = scaler.fit_transform(y_aligned.reshape(-1, 1)).flatten()

    # DTW 거리
    distance = dtw_distance(normalized_x, normalized_y)

    # 길이로 정규화
    normalized_distance = distance / len(normalized_x)
    return normalized_distance

def find_comovement_pairs(
    pivot,
    max_lag=9,
    min_nonzero=12,
    corr_threshold=0.4,
    score_threshold=45,
    use_dtw=True,              # DTW 사용 여부
    dtw_threshold=2.0          # DTW 임계값
):
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)
    results = []

    for i, leader in tqdm(enumerate(items), total=len(items), desc="Finding pairs"):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        candidates = []

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0
            best_p_value = 1.0
            best_dtw = float('inf')

            # lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                corr, p_value = safe_corr_with_pvalue(x[:-lag], y[lag:])

                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag
                    best_p_value = p_value

            # 1단계: 상관계수 임계값 통과 시
            if best_lag is not None and abs(best_corr) >= corr_threshold:

                # 2단계: DTW 검증 (옵션)
                if use_dtw:
                    best_dtw = calculate_normalized_dtw(x, y, best_lag)

                    # DTW 임계값 통과 못하면 스킵
                    if best_dtw > dtw_threshold:
                        continue

                # 종합 점수 계산
                score = calculate_score_advanced(
                    best_corr, best_p_value, best_lag
                )

                # DTW 점수 반영 (옵션)
                if use_dtw and best_dtw != float('inf'):
                    # DTW가 낮을수록 좋으므로 역수 사용
                    dtw_bonus = 1 / (1 + best_dtw)
                    score = score * (1 + dtw_bonus)  # DTW 보너스 추가

                # 점수 임계값 통과 시 후보에 추가
                if score >= score_threshold:
                    candidates.append({
                        "following_item_id": follower,
                        "best_lag": best_lag,
                        "max_corr": best_corr,
                        "p_value": best_p_value,
                        "dtw_distance": best_dtw if use_dtw else None,
                        "comovement_score": score
                    })

        # 점수 순으로 정렬 (높은 순)
        candidates.sort(key=lambda x: -x['comovement_score'])

        for candidate in candidates:
            result_dict = {
                "leading_item_id": leader,
                "following_item_id": candidate["following_item_id"],
                "best_lag": candidate["best_lag"],
                "max_corr": candidate["max_corr"]
            }

            # DTW 사용 시 추가 정보 포함
            if use_dtw:
                result_dict["dtw_distance"] = candidate["dtw_distance"]
                result_dict["comovement_score"] = candidate["comovement_score"]

            results.append(result_dict)

    pairs = pd.DataFrame(results)
    return pairs

# 사용 예시

# # DTW 없이 (기존 방식)
# pairs_basic = find_comovement_pairs(
#     pivot,
#     max_lag=9,
#     min_nonzero=12,
#     corr_threshold=0.4,
#     score_threshold=35,
#     use_dtw=False
# )

# DTW 추가 (상관계수 + DTW 하이브리드)
pairs_with_dtw= find_comovement_pairs(
    pivot,
    max_lag=9,
    min_nonzero=12,
    corr_threshold=0.33,      # 1차 필터: 상관계수
    score_threshold=40,
    use_dtw=True,            # DTW 활성화
    dtw_threshold=1.0       # 2차 필터: DTW 거리
)

# print(f"Basic pairs: {len(pairs_basic)}")
print(f"Pairs with DTW: {len(pairs_with_dtw)}")
print("\n=== Top 10 pairs with DTW ===")
print(pairs_with_dtw.head(20))

Finding pairs: 100%|██████████| 100/100 [00:46<00:00,  2.13it/s]

Pairs with DTW: 3368

=== Top 10 pairs with DTW ===
   leading_item_id following_item_id  best_lag  max_corr  dtw_distance  \
0         AANGBULD          ZCELVYQU         7  0.677701      0.304327   
1         AANGBULD          ZKENOUDA         1  0.599969      0.513452   
2         AANGBULD          DEWLVASR         6  0.640221      0.490232   
3         AANGBULD          NAQIHUKZ         2  0.525490      0.471456   
4         AANGBULD          GKQIJYDH         6  0.582501      0.464203   
5         AANGBULD          FTSVTTSR         3  0.531400      0.519376   
6         AANGBULD          VUAFAIYJ         2  0.509968      0.671474   
7         AANGBULD          KJNSOAHR         9  0.575636      0.454634   
8         AANGBULD          ZXERAXWP         9  0.569322      0.514398   
9         AANGBULD          XIIEJNEE         7  0.526602      0.436966   
10        AANGBULD          LLHREMKS         5  0.499734      0.485700   
11        AANGBULD          ELQGMQWE         8  0.474939    

In [8]:
# ============================================================
#  ⬇⬇⬇⬇⬇⬇⬇⬇  여기에 넣어라: Pairs 생성 직후 위치  ⬇⬇⬇⬇⬇⬇⬇⬇
# ============================================================

# 1) pivot 기반 전체 item graph 생성
import networkx as nx
import numpy as np
from community import community_louvain  # Louvain

# --- 그래프 생성 ---
def build_item_graph(pivot_value, corr_thr=0.3, lag_thr=0.2, max_lag=3):

    items = pivot_value.columns.tolist()
    G = nx.DiGraph()

    # Pearson corr
    corr_matrix = pivot_value.corr()

    # lagged corr
    def lag_corr(a, b):
        a = a - a.mean()
        b = b - b.mean()
        best = 0
        for lag in range(1, max_lag + 1):
            if len(a) <= lag: break
            c = np.corrcoef(a[:-lag], b[lag:])[0, 1]
            best = max(best, abs(c))
        return best

    for i in items:
        for j in items:
            if i == j: continue
            w1 = corr_matrix.loc[i, j]
            if abs(w1) >= corr_thr:
                G.add_edge(i, j, weight=float(w1))

            w2 = lag_corr(pivot_value[i].values, pivot_value[j].values)
            if w2 >= lag_thr:
                G.add_edge(i, j, weight=float(w2))

    return G


# pivot_value는 year_month × item_id 형태이므로 다시 transpose
pivot_value_for_graph = pivot_value.copy()

G = build_item_graph(pivot_value_for_graph)

print("그래프 노드수:", G.number_of_nodes())
print("그래프 엣지수:", G.number_of_edges())


# --- Louvain Community ---
partition = community_louvain.best_partition(G.to_undirected(), weight='weight')
louvain_df = pd.DataFrame({'item_id': list(partition.keys()),
                           'community': list(partition.values())})


# --- Greedy Modularity ---
greedy_com = nx.algorithms.community.greedy_modularity_communities(G.to_undirected())
greedy_assign = {}
for cid, group in enumerate(greedy_com):
    for node in group:
        greedy_assign[node] = cid

greedy_df = pd.DataFrame({'item_id': list(greedy_assign.keys()),
                          'community': list(greedy_assign.values())})


# --- Centrality ---
in_deg  = dict(G.in_degree(weight='weight'))
out_deg = dict(G.out_degree(weight='weight'))
close   = nx.closeness_centrality(G)
btwn    = nx.betweenness_centrality(G, weight='weight')

centrality_df = pd.DataFrame({
    'item_id': list(G.nodes()),
    'in_degree': [in_deg.get(i, 0) for i in G.nodes()],
    'out_degree': [out_deg.get(i, 0) for i in G.nodes()],
    'closeness': [close.get(i, 0) for i in G.nodes()],
    'betweenness': [btwn.get(i, 0) for i in G.nodes()]
})


# -------------------------
# 2) pair_df에 feature merge
# -------------------------

# dict 형태로 변환
louvain_map = dict(zip(louvain_df['item_id'], louvain_df['community']))
greedy_map  = dict(zip(greedy_df['item_id'], greedy_df['community']))
cent_map    = centrality_df.set_index('item_id').to_dict(orient='index')


def add_graph_features(row):

    A = row.leading_item_id
    B = row.following_item_id

    # Louvain
    cA = louvain_map.get(A, -1)
    cB = louvain_map.get(B, -1)

    # Greedy
    gA = greedy_map.get(A, -1)
    gB = greedy_map.get(B, -1)

    # centrality
    CA = cent_map.get(A, {'in_degree':0,'out_degree':0,'closeness':0,'betweenness':0})
    CB = cent_map.get(B, {'in_degree':0,'out_degree':0,'closeness':0,'betweenness':0})

    return pd.Series({
        'lead_comm': cA,
        'foll_comm': cB,
        'same_comm': int(cA == cB),

        'lead_greedy': gA,
        'foll_greedy': gB,
        'same_greedy': int(gA == gB),

        'lead_in_deg': CA['in_degree'],
        'foll_in_deg': CB['in_degree'],
        'deg_gap': CA['in_degree'] - CB['in_degree'],
        'deg_ratio': CA['in_degree'] / (CB['in_degree'] + 1e-6),

        'lead_btwn': CA['betweenness'],
        'foll_btwn': CB['betweenness'],
        'btwn_gap': CA['betweenness'] - CB['betweenness'],
        'btwn_ratio': CA['betweenness'] / (CB['betweenness'] + 1e-6),
    })


graph_feats = pairs_with_dtw.apply(add_graph_features, axis=1)
pairs_with_dtw = pd.concat([pairs_with_dtw, graph_feats], axis=1)

print("그래프 기반 feature 추가 완료!")
print(pairs_with_dtw.head())

그래프 노드수: 100
그래프 엣지수: 5375
그래프 기반 feature 추가 완료!
  leading_item_id following_item_id  best_lag  max_corr  dtw_distance  \
0        AANGBULD          ZCELVYQU         7  0.677701      0.304327   
1        AANGBULD          ZKENOUDA         1  0.599969      0.513452   
2        AANGBULD          DEWLVASR         6  0.640221      0.490232   
3        AANGBULD          NAQIHUKZ         2  0.525490      0.471456   
4        AANGBULD          GKQIJYDH         6  0.582501      0.464203   

   comovement_score  lead_comm  foll_comm  same_comm  lead_greedy  \
0        120.909908        0.0        0.0        1.0          1.0   
1        119.569775        0.0        0.0        1.0          1.0   
2        114.993153        0.0        0.0        1.0          1.0   
3        112.170840        0.0        0.0        1.0          1.0   
4        109.014231        0.0        0.0        1.0          1.0   

   foll_greedy  same_greedy  lead_in_deg  foll_in_deg    deg_gap  deg_ratio  \
0          1.0    

# 기존 추출된 pair들, 네트워크 극단값 기반으로 저품질 제거하고 모델링에 넣기 (옵션2 - 데이터 품질 향상)

In [9]:
### 안쓸땐 주석처리

# === (A) 그래프 기반 재가중치 + pair 필터링 (Stage1 품질 개선) ===
import numpy as np

# 그래프 피처까지 merge가 끝난 최종 pair 테이블
# columns 예시: leading_item_id, following_item_id, best_lag, max_corr,
#               dtw_distance, comovement_score,
#               lead_comm, foll_comm, same_comm,
#               lead_greedy, foll_greedy, same_greedy,
#               lead_in_deg, foll_in_deg, deg_gap, deg_ratio,
#               lead_btwn, foll_btwn, btwn_gap, btwn_ratio
pairs_for_model = pairs_with_dtw.copy()

# 1) deg_ratio, btwn_ratio 이상치/극단값 클리핑
deg_ratio_clipped = pairs_for_model['deg_ratio'].clip(0.3, 3.0)
btwn_ratio_raw = pairs_for_model['btwn_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
btwn_ratio_clipped = btwn_ratio_raw.clip(0.3, 3.0)

# 2) 그래프 기반 가중 score 정의
#    - 기본: comovement_score
#    - same_comm == 1 인 쌍에 약간 보너스
#    - deg_ratio, btwn_ratio는 너무 세게 쓰지 않고 약한 지수로 반영
pairs_for_model['graph_score'] = (
    pairs_for_model['comovement_score']
    * (1.05 + 0.05 * pairs_for_model['same_comm'])     # same_comm 1이면 대략 10% 보너스
    * (deg_ratio_clipped / 1.5) ** 0.3                 # degree 비례 약하게 반영
    * (btwn_ratio_clipped) ** 0.2                      # betweenness 비례 아주 약하게 반영
)

# 3) leader별 상위 TOP_K 개 pair만 사용 (노이즈 쌍 컷)
TOP_K = 35  # 필요하면 바꿔보면서 실험

pairs_for_model = (
    pairs_for_model
    .sort_values(['leading_item_id', 'graph_score'], ascending=[True, False])
    .groupby('leading_item_id')
    .head(TOP_K)
    .reset_index(drop=True)
)

print("그래프 노드수는 그대로, 재가중치 후 사용될 pair 수:", len(pairs_for_model))
pairs_for_model.head()

그래프 노드수는 그대로, 재가중치 후 사용될 pair 수: 2888


,leading_item_id,following_item_id,best_lag,max_corr,dtw_distance,comovement_score,lead_comm,foll_comm,same_comm,lead_greedy,...,same_greedy,lead_in_deg,foll_in_deg,deg_gap,deg_ratio,lead_btwn,foll_btwn,btwn_gap,btwn_ratio,graph_score
0,AANGBULD,NAQIHUKZ,2,0.525490,0.471456,112.170840,0.0,0.0,1.0,1.0,...,1.0,20.448407,8.145517,12.302890,2.510388,0.003504,0.000000,0.003504,3504.432076,179.387328
1,AANGBULD,GKQIJYDH,6,0.582501,0.464203,109.014231,0.0,0.0,1.0,1.0,...,1.0,20.448407,11.049129,9.399278,1.850680,0.003504,0.000000,0.003504,3504.432076,159.100730
2,AANGBULD,FTSVTTSR,3,0.531400,0.519376,106.450898,0.0,0.0,1.0,1.0,...,1.0,20.448407,11.656946,8.791462,1.754182,0.003504,0.000722,0.002783,4.850420,152.883738
3,AANGBULD,ZKENOUDA,1,0.599969,0.513452,119.569775,0.0,0.0,1.0,1.0,...,0.0,20.448407,25.388316,-4.939909,0.805426,0.003504,0.000825,0.002680,4.244852,135.962344
4,AANGBULD,ZCELVYQU,7,0.677701,0.304327,120.909908,0.0,0.0,1.0,1.0,...,1.0,20.448407,15.473195,4.975213,1.321537,0.003504,0.002886,0.000618,1.213865,133.102167


In [ ]:
pairs_for_model

,leading_item_id,following_item_id,best_lag,max_corr,dtw_distance,comovement_score,lead_comm,foll_comm,same_comm,lead_greedy,...,same_greedy,lead_in_deg,foll_in_deg,deg_gap,deg_ratio,lead_btwn,foll_btwn,btwn_gap,btwn_ratio,graph_score
0,AANGBULD,NAQIHUKZ,2,0.525490,0.471456,112.170840,0.0,3.0,0.0,1.0,...,1.0,20.448407,8.145517,12.302890,2.510388,0.003504,0.000000,0.003504,3504.432076,171.233359
1,AANGBULD,GKQIJYDH,6,0.582501,0.464203,109.014231,0.0,0.0,1.0,1.0,...,1.0,20.448407,11.049129,9.399278,1.850680,0.003504,0.000000,0.003504,3504.432076,159.100730
2,AANGBULD,FTSVTTSR,3,0.531400,0.519376,106.450898,0.0,3.0,0.0,1.0,...,1.0,20.448407,11.656946,8.791462,1.754182,0.003504,0.000722,0.002783,4.850420,145.934477
3,AANGBULD,ZKENOUDA,1,0.599969,0.513452,119.569775,0.0,0.0,1.0,1.0,...,0.0,20.448407,25.388316,-4.939909,0.805426,0.003504,0.000825,0.002680,4.244852,135.962344
4,AANGBULD,ZCELVYQU,7,0.677701,0.304327,120.909908,0.0,0.0,1.0,1.0,...,1.0,20.448407,15.473195,4.975213,1.321537,0.003504,0.002886,0.000618,1.213865,133.102167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2883,ZXERAXWP,YSYHGLQK,1,0.374210,0.571980,75.581012,0.0,3.0,0.0,0.0,...,0.0,16.549164,14.173036,2.376128,1.167651,0.304679,0.890229,-0.585549,0.342248,59.406884
2884,ZXERAXWP,STZDBITS,7,0.361019,0.624989,58.592246,0.0,3.0,0.0,0.0,...,0.0,16.549164,11.655973,4.893191,1.419801,0.304679,0.593177,-0.288497,0.513639,52.966587
2885,ZXERAXWP,KJNSOAHR,7,0.370593,0.673319,59.018251,0.0,0.0,1.0,0.0,...,1.0,16.549164,22.532921,-5.983758,0.734444,0.304679,0.350752,-0.046073,0.868643,50.945690
2886,ZXERAXWP,XUOIQPFL,1,-0.353922,0.863469,68.802086,0.0,1.0,0.0,0.0,...,1.0,16.549164,23.796487,-7.247324,0.695446,0.304679,0.810864,-0.506184,0.375746,47.165173


In [ ]:
print(len(pairs_with_dtw)), print(len(pairs_for_model))

3368
2888


(None, None)



---





---



In [ ]:
import os
import datetime

# result 폴더 생성 (이미 있으면 무시)
os.makedirs('./result', exist_ok=True)
date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 파일 저장
file_name = f'./result/sy_{date_str}.csv'
submission.to_csv(file_name, index=False)

## ✅ 조건부 Weight 피처 추가

**검증 결과:**
- Value 작을 때: Weight-Value 상관 0.254 (약함)
- Value 클 때: Weight-Value 상관 0.640 (강함)

**→ Value 클 때만 Weight 강조!**

# 이 바로 밑에 있는 코드 쓰면 돼요!!! 0.4093328789
# 옵션1 - 네트워크 구조로 pair를 필터링 한다음 학습하는 코드 (성능개선됨, 0.369대)
#제안
여기서 구조는 그대로 유지하면서 안정적으로 점수 올릴 수 있는 방향
현재는 그대로 value 예측
무역량처럼 분포가 한쪽으로 치우치고 극단적으로 큰 값이 섞여 있는 경우 log1p(target)으로 학습하고 예측 후에 expm1로 되돌리는 게 더 쉬운 분포를 만들어주는 경우가 많음
#target에 log를 취해줬더니 성능이 더 좋아짐
0.4093328789


In [15]:
import numpy as np
import pandas as pd

def summarize_follower_conflict(submission: pd.DataFrame, title=""):
    df = submission.copy()
    g = df.groupby("following_item_id")["value"].agg(n="size", nunique="nunique")

    print("\n" + "=" * 80)
    print(f"🔎 follower 중복/불일치 요약 {title}")
    print("=" * 80)
    print(f"- 전체 row: {len(df):,}")
    print(f"- follower 개수(유니크): {g.shape[0]:,}")
    print(f"- follower당 평균 row 수: {g['n'].mean():.2f}")
    print(f"- follower당 중앙값 row 수: {g['n'].median():.0f}")
    print(f"- value가 2개 이상인 follower 수: {(g['nunique'] > 1).sum():,}")
    print("  (상위 5개 follower row 수)")
    print(g.sort_values("n", ascending=False).head(5))


def follower_winsorize_clip(submission: pd.DataFrame, q=0.02, min_group=8, use_log=True):
    """
    follower별 극단값만 '클리핑'하는 안전 후처리.
    - q=0.02면 각 follower 그룹에서 상/하위 2%를 경계값으로 잘라냄
    - min_group 이상인 follower 그룹에만 적용(표본 적으면 건드리면 불안정)
    - use_log=True 권장(로그공간에서 클리핑하면 극단값에 덜 민감)
    """
    df = submission.copy()

    y = df["value"].astype(float).clip(lower=0.0).values
    y_space = np.log1p(y) if use_log else y.copy()
    y_new = y_space.copy()

    groups = df.groupby("following_item_id").groups
    for fid, idx in groups.items():
        idx = np.fromiter(idx, dtype=int)  # 빠르고 깔끔
        if len(idx) < min_group:
            continue

        vals = y_space[idx]
        lo = np.quantile(vals, q)
        hi = np.quantile(vals, 1 - q)
        y_new[idx] = np.clip(vals, lo, hi)

    y_back = np.expm1(y_new) if use_log else y_new
    df["value"] = np.maximum(0.0, np.round(y_back)).astype(int)
    return df


def compare_before_after(before: pd.DataFrame, after: pd.DataFrame, top=10):
    m = before.merge(after, on=["leading_item_id", "following_item_id"], suffixes=("_before", "_after"))
    changed = (m["value_before"] != m["value_after"]).sum()

    print("\n" + "=" * 80)
    print("✅ 후처리 적용 결과 체크")
    print("=" * 80)
    print(f"- 바뀐 row 수: {changed:,} / {len(m):,} ({changed/len(m)*100:.2f}%)")

    if changed > 0:
        lb = np.log1p(m["value_before"].clip(lower=0))
        la = np.log1p(m["value_after"].clip(lower=0))
        delta = (la - lb).abs()

        print(f"- |log 변화| 평균: {delta.mean():.4f}, 중앙값: {delta.median():.4f}, 최대: {delta.max():.4f}")
        print("\n(변화가 큰 상위 샘플)")
        show = m.assign(abs_log_change=delta).sort_values("abs_log_change", ascending=False).head(top)
        print(show[["leading_item_id", "following_item_id", "value_before", "value_after", "abs_log_change"]])


In [16]:
"""
무역 데이터 공행성 예측 모델 (2,700개 목표 최적화)
- 원본: 3,368개
- 목표: 2,700개 (약 80% 유지)
- 적절한 노이즈 제거 (중간 모드)
- 32개 피처 + LGBM/XGB 앙상블
"""

import datetime
import pandas as pd
import numpy as np
from scipy import stats
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from tqdm import tqdm

# ============================================================================
# 데이터 로드
# ============================================================================

print("="*80)
print("데이터 로드")
print("="*80)

monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({'value': 'sum', 'weight': 'sum'})
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

pivot = monthly.pivot(index="item_id", columns="ym", values="value").fillna(0.0)
pivot_weight = monthly.pivot(index="item_id", columns="ym", values="weight").fillna(0.0)
months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료: {len(pivot)}개 품목, {len(months_dt)}개월")

# ============================================================================
# ⭐ 중간 품질 점수 계산 (2,700개 목표용)
# ============================================================================

def calculate_quality_score_moderate(a_series, b_series, lag):
    """
    중간 품질 점수 계산 (0-100점)
    - 원본 3,368개 → 목표 2,700개
    - 적절한 노이즈 제거 (80% 유지)
    """

    scores = {}

    # 1. 0 비율 (10점) - 중간
    zero_ratio_a = (a_series == 0).sum() / len(a_series)
    zero_ratio_b = (b_series == 0).sum() / len(b_series)
    max_zero = max(zero_ratio_a, zero_ratio_b)

    if max_zero < 0.17:
        scores['zero'] = 10
    elif max_zero < 0.37:
        scores['zero'] = 7
    elif max_zero < 0.57:  # 57%까지 허용
        scores['zero'] = 4
    else:
        scores['zero'] = 0

    # 2. 변동계수 (10점) - 중간
    cv_a = np.std(a_series) / (np.mean(a_series) + 1)
    cv_b = np.std(b_series) / (np.mean(b_series) + 1)
    max_cv = max(cv_a, cv_b)

    if max_cv < 0.75:
        scores['cv'] = 10
    elif max_cv < 1.4:
        scores['cv'] = 7
    elif max_cv < 2.7:  # 2.7까지 허용
        scores['cv'] = 4
    else:
        scores['cv'] = 0

    # 3. 스파이크 (10점) - 중간
    a_changes = np.abs(np.diff(a_series))
    b_changes = np.abs(np.diff(b_series))

    if len(a_changes) > 0 and len(b_changes) > 0:
        a_spike = a_changes.max() / (np.median(a_changes) + 1)
        b_spike = b_changes.max() / (np.median(b_changes) + 1)
        max_spike = max(a_spike, b_spike)

        if max_spike < 4.5:
            scores['spike'] = 10
        elif max_spike < 9.5:
            scores['spike'] = 7
        elif max_spike < 17:  # 17까지 허용
            scores['spike'] = 4
        else:
            scores['spike'] = 0
    else:
        scores['spike'] = 5

    # 4. 안정성 (15점) - 중간
    n = len(a_series)

    try:
        mid = n // 2

        if mid >= 12 and mid + lag < len(b_series):
            corr_first = np.corrcoef(
                a_series[:mid],
                b_series[lag:mid+lag]
            )[0, 1]

            corr_second = np.corrcoef(
                a_series[mid:],
                b_series[mid+lag:]
            )[0, 1]

            if not np.isnan(corr_first) and not np.isnan(corr_second):
                consistency = 1 - abs(corr_first - corr_second)

                if consistency > 0.52:  # 중간
                    scores['stability'] = 15
                elif consistency > 0.35:
                    scores['stability'] = 10
                elif consistency > 0.22:
                    scores['stability'] = 6
                else:
                    scores['stability'] = 3
            else:
                scores['stability'] = 5
        else:
            scores['stability'] = 5
    except:
        scores['stability'] = 5

    # 5. 트렌드 일치 (10점) - 중간
    if len(a_series) > 0 and len(b_series) > 0:
        x = np.arange(len(a_series))

        if np.std(a_series) > 0:
            slope_a, _ = np.polyfit(x, a_series, 1)
        else:
            slope_a = 0

        if np.std(b_series) > 0:
            slope_b, _ = np.polyfit(x, b_series, 1)
        else:
            slope_b = 0

        if (slope_a > 0 and slope_b > 0) or (slope_a < 0 and slope_b < 0):
            ratio = abs(slope_a) / (abs(slope_b) + 1e-10)
            if 0.38 < ratio < 2.7:  # 중간
                scores['trend'] = 10
            elif 0.25 < ratio < 4.2:
                scores['trend'] = 6
            else:
                scores['trend'] = 3
        elif slope_a == 0 or slope_b == 0:
            scores['trend'] = 5
        else:
            scores['trend'] = 2
    else:
        scores['trend'] = 4

    # 6. 방향 일치도 (10점) - 중간
    if len(a_changes) > lag and len(b_changes) > lag:
        direction_match = (np.sign(a_changes[:-lag]) == np.sign(b_changes[lag:])).mean()

        if direction_match > 0.64:  # 중간
            scores['direction'] = 10
        elif direction_match > 0.54:
            scores['direction'] = 7
        elif direction_match > 0.44:
            scores['direction'] = 4
        else:
            scores['direction'] = 1
    else:
        scores['direction'] = 4

    # 7. 통계적 유의성 (10점) - 중간
    try:
        if len(a_series) > lag:
            _, p_value = stats.pearsonr(
                a_series[:-lag] if lag > 0 else a_series,
                b_series[lag:] if lag > 0 else b_series
            )

            if p_value < 0.007:  # 중간
                scores['significance'] = 10
            elif p_value < 0.025:
                scores['significance'] = 7
            elif p_value < 0.07:
                scores['significance'] = 4
            else:
                scores['significance'] = 1
        else:
            scores['significance'] = 4
    except:
        scores['significance'] = 4

    # 8. 자기상관 (10점) - 중간
    try:
        if len(b_series) > 1:
            autocorr = np.corrcoef(b_series[:-1], b_series[1:])[0, 1]

            if 0.24 < autocorr < 0.86:  # 중간
                scores['autocorr'] = 10
            elif 0.09 < autocorr < 0.91:
                scores['autocorr'] = 7
            else:
                scores['autocorr'] = 3
        else:
            scores['autocorr'] = 4
    except:
        scores['autocorr'] = 4

    # 9. 이상치 비율 (10점) - 중간
    def count_outliers(series):
        q1 = np.percentile(series, 25)
        q3 = np.percentile(series, 75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        return outliers / len(series)

    outlier_ratio_a = count_outliers(a_series)
    outlier_ratio_b = count_outliers(b_series)
    max_outlier = max(outlier_ratio_a, outlier_ratio_b)

    if max_outlier < 0.085:
        scores['outlier'] = 10
    elif max_outlier < 0.16:
        scores['outlier'] = 7
    elif max_outlier < 0.26:  # 26%까지 허용
        scores['outlier'] = 4
    else:
        scores['outlier'] = 1

    # 10. 분포 정규성 (5점) - 중간
    try:
        if len(b_series) >= 8:
            _, p_shapiro = stats.shapiro(b_series)

            if p_shapiro > 0.02:  # 중간
                scores['normality'] = 5
            elif p_shapiro > 0.004:
                scores['normality'] = 3
            else:
                scores['normality'] = 1
        else:
            scores['normality'] = 2
    except:
        scores['normality'] = 2

    total_score = sum(scores.values())

    return total_score, scores


# ============================================================================
# ⭐ 2,700개 목표 노이즈 제거
# ============================================================================

def remove_noise_pairs_for_2700(pairs, pivot, target_count=2700):
    """
    2,700개 목표 노이즈 제거

    원본: 3,368개
    목표: 2,700개 (약 80% 유지)
    전략: 적절한 노이즈 제거 (중간 모드)
    """

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 노이즈 제거 (중간 모드)")
    print("="*80)
    print(f"원본: {len(pairs):,}개")
    print(f"목표: {target_count:,}개 (약 {target_count/len(pairs)*100:.0f}% 유지)")

    quality_scores = []
    detailed_scores = []

    # 품질 점수 계산 (중간 기준)
    print(f"\n품질 평가 중...")
    for row in tqdm(pairs.itertuples(), total=len(pairs), desc="평가"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            quality_scores.append(0)
            detailed_scores.append({})
            continue

        a_series = pivot.loc[leader].values
        b_series = pivot.loc[follower].values

        total_score, scores = calculate_quality_score_moderate(a_series, b_series, lag)

        quality_scores.append(total_score)
        detailed_scores.append(scores)

    # 품질 점수 추가
    pairs_with_score = pairs.copy()
    pairs_with_score['quality_score'] = quality_scores

    # 목표 개수 기반 임계값 계산
    sorted_scores = sorted(quality_scores, reverse=True)

    if len(sorted_scores) >= target_count:
        # 목표 개수번째 점수를 임계값으로
        threshold = sorted_scores[target_count - 1]
        # 5점 단위로 내림
        threshold = max(33, (threshold // 5) * 5)  # 최소 33점
    else:
        threshold = 33  # 최소 임계값

    # 필터링
    pairs_clean = pairs_with_score[
        pairs_with_score['quality_score'] >= threshold
    ].copy()

    # 품질 점수 컬럼 제거
    pairs_for_model = pairs_clean.drop(columns=['quality_score'], errors='ignore')

    # 통계
    removed = len(pairs) - len(pairs_clean)
    removed_ratio = removed / len(pairs) * 100
    kept_ratio = len(pairs_clean) / len(pairs) * 100

    print(f"\n{'='*80}")
    print(f"✅ 노이즈 제거 완료!")
    print(f"{'='*80}")
    print(f"원본: {len(pairs):,}개")
    print(f"제거: {removed:,}개 ({removed_ratio:.1f}%)")
    print(f"유지: {len(pairs_clean):,}개 ({kept_ratio:.1f}%)")
    print(f"임계값: {threshold}점")

    # 점수 분포
    print(f"\n품질 점수 분포:")
    print(f"  - 평균: {np.mean(quality_scores):.1f}점")
    print(f"  - 중앙값: {np.median(quality_scores):.1f}점")
    print(f"  - 최소: {np.min(quality_scores):.1f}점")
    print(f"  - 최대: {np.max(quality_scores):.1f}점")

    # 선택된 쌍의 점수
    selected_scores = pairs_clean['quality_score'].values
    print(f"\n선택된 쌍의 점수:")
    print(f"  - 평균: {np.mean(selected_scores):.1f}점")
    print(f"  - 최소: {np.min(selected_scores):.1f}점")

    # 구간별 분포
    print(f"\n점수 구간별 분포:")
    bins = [0, 30, 40, 50, 60, 70, 80, 90, 100]
    for i in range(len(bins)-1):
        total_count = sum((np.array(quality_scores) >= bins[i]) & (np.array(quality_scores) < bins[i+1]))
        selected_count = sum((selected_scores >= bins[i]) & (selected_scores < bins[i+1]))
        print(f"  {bins[i]:3d}-{bins[i+1]:3d}점: 전체 {total_count:4,}개, 선택 {selected_count:4,}개")

    # 기준별 평균 점수
    print(f"\n기준별 평균 점수 (선택된 쌍):")
    criteria_names = {
        'zero': '0 비율',
        'cv': '변동계수',
        'spike': '스파이크',
        'stability': '안정성',
        'trend': '트렌드 일치',
        'direction': '방향 일치도',
        'significance': '통계적 유의성',
        'autocorr': '자기상관',
        'outlier': '이상치 비율',
        'normality': '정규성'
    }

    selected_indices = pairs_with_score[pairs_with_score['quality_score'] >= threshold].index
    selected_detailed = [detailed_scores[i] for i in selected_indices if i < len(detailed_scores)]

    for key, name in criteria_names.items():
        scores = [s.get(key, 0) for s in selected_detailed if s]
        if scores:
            if key == 'stability':
                max_score = 15
            elif key == 'normality':
                max_score = 5
            else:
                max_score = 10

            print(f"  {name:15s}: 평균 {np.mean(scores):5.1f}점 (최대 {max_score}점)")

    return pairs_for_model


# ============================================================================
# 조건부 Weight 피처
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """조건부 Weight 피처 4개"""
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    weight_value_ratio = weight * (value / (value_median + 1))
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    if value > value_q75:
        weight_importance = weight * 1.5
    elif value > value_median:
        weight_importance = weight * 1.0
    else:
        weight_importance = weight * 0.1

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }


# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt):
    """32개 피처 학습 데이터 생성"""
    months = months_dt
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        for t in range(max(lag, 12), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]
            b_weight_t = b_weight_series[t]
            target_month = months[t + 1].month


            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0
            lag_weight = 1 / (1 + lag)

            b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

            rows.append({
                "b_t": b_t, "b_t_1": b_t_1, "b_t_12": b_t_12, "a_t_lag": a_t_lag,
                "max_corr": corr, "best_lag": float(lag), "month": float(target_month),
                "a_roll_mean_3": a_roll_mean_3,
                "b_diff": b_diff, "b_pct_change": b_pct_change, "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3, "b_roll_mean_6": b_roll_mean_6, "b_roll_std_3": b_roll_std_3,
                "ab_ratio": ab_ratio, "ab_diff": ab_diff, "corr_weighted_a": corr_weighted_a,
                "b_cv": b_cv, "b_acceleration": b_acceleration, "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio, "b_percentile_rank": b_percentile_rank,
                "quarter": float(quarter), "is_year_end": is_year_end, "lag_weight": lag_weight,
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                "target": np.log1p(b_t_plus_1),
            })

    return pd.DataFrame(rows)


# ============================================================================
# 예측 함수
# ============================================================================

def predict_optimized(pivot, pivot_weight, pairs, model_lgb, model_xgb, months_dt):
    """32개 피처 예측 + 앙상블"""
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []
    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    for row in tqdm(pairs.itertuples(index=False), desc="예측"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag < 0 or t_last < 12:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]
        b_weight_t = b_weight_series[t_last]


        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0
        lag_weight = 1 / (1 + lag)

        b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

        X_test = np.array([[
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end, lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance']
        ]])

        pred_lgb = model_lgb.predict(X_test)[0]
        pred_xgb = model_xgb.predict(X_test)[0]
        y_pred_log = 0.7 * pred_lgb + 0.3 * pred_xgb
        y_pred = np.expm1(y_pred_log)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)

# ============================================================================
# 메인 실행
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 최적화 버전")
    print("="*80)

    # ⭐ 2,700개 목표 노이즈 제거
    pairs_for_model = remove_noise_pairs_for_2700(
        pairs=pairs_with_dtw,
        pivot=pivot,
        target_count=2700      # ⭐ 목표: 2,700개
    )

    print(f"\n최종 사용 쌍: {len(pairs_for_model)}개")

    # 학습
    print("\n" + "="*80)
    print("학습 데이터 생성")
    print("="*80)

    df_train_model = build_training_data_optimized(pivot, pivot_weight, pairs_for_model, months_dt)
    print(f"✓ 학습 데이터: {df_train_model.shape}")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth', 'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end', 'lag_weight',
        'b_weight_for_high_value', 'b_weight_value_ratio', 'b_exp_weighted', 'b_weight_importance'
    ]

    if not df_train_model.empty:
        print("\n" + "="*80)
        print("모델 학습")
        print("="*80)

        train_X = df_train_model[feature_cols].values
        train_y = df_train_model["target"].values

        print(f"학습 샘플: {len(train_X):,}, 피처: {len(feature_cols)}")

        model_lgb = LGBMRegressor(
            random_state=42, n_estimators=700, learning_rate=0.01, max_depth=8,
            num_leaves=50, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, min_split_gain=0.01
        )

        print("\nLGBM 학습 중...")
        model_lgb.fit(train_X, train_y)
        print("✓ LGBM 완료")

        model_xgb = XGBRegressor(
            random_state=42, n_estimators=900, learning_rate=0.02, max_depth=8,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
            tree_method='hist'
        )

        print("XGBoost 학습 중...")
        model_xgb.fit(train_X, train_y)
        print("✓ XGBoost 완료")

        print("\n" + "="*80)
        print("예측")
        print("="*80)

        submission = predict_optimized(pivot, pivot_weight, pairs_for_model, model_lgb, model_xgb, months_dt)
        print(f"\n✓ 예측 완료: {len(submission)}개")


         #(1) 후처리 전 상태 확인
        summarize_follower_conflict(submission, title="(후처리 전)")

        #(2) follower별 극단값만 클리핑(안전 후처리)
        submission_pp = follower_winsorize_clip(
            submission,
            q=0.02,
            min_group=8,
            use_log=True
        )
        #(3) 후처리 후 상태 확인 + 얼마나 바뀌었는지 체크
        summarize_follower_conflict(submission_pp, title="(후처리 후 : clip)")
        compare_before_after(submission, submission_pp, top=10)

        submission = submission_pp
        date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        submission.to_csv(f'_result_sy_2700_moderate_{date_str}.csv', index=False)
        print("\n✅ 2,700개 목표 버전 완료!")
        print("  - 원본: 3,368개")
        print("  - 목표: 2,700개")
        print("  - 중간 기준 (적절한 노이즈 제거)")
        print("  - 32개 피처")
        print("  - LGBM + XGB 앙상블")
        print("="*80)

데이터 로드
✓ 데이터 로드 완료: 100개 품목, 43개월

⭐ 2,700개 목표 최적화 버전

⭐ 2,700개 목표 노이즈 제거 (중간 모드)
원본: 3,368개
목표: 2,700개 (약 80% 유지)

품질 평가 중...


평가: 100%|██████████| 3368/3368 [00:12<00:00, 276.11it/s]



✅ 노이즈 제거 완료!
원본: 3,368개
제거: 611개 (18.1%)
유지: 2,757개 (81.9%)
임계값: 55점

품질 점수 분포:
  - 평균: 65.5점
  - 중앙값: 67.0점
  - 최소: 36.0점
  - 최대: 90.0점

선택된 쌍의 점수:
  - 평균: 69.2점
  - 최소: 55.0점

점수 구간별 분포:
    0- 30점: 전체    0개, 선택    0개
   30- 40점: 전체   29개, 선택    0개
   40- 50점: 전체  205개, 선택    0개
   50- 60점: 전체  708개, 선택  331개
   60- 70점: 전체 1,002개, 선택 1,002개
   70- 80점: 전체 1,212개, 선택 1,212개
   80- 90점: 전체  208개, 선택  208개
   90-100점: 전체    4개, 선택    4개

기준별 평균 점수 (선택된 쌍):
  0 비율           : 평균   9.8점 (최대 10점)
  변동계수           : 평균   7.8점 (최대 10점)
  스파이크           : 평균   6.2점 (최대 10점)
  안정성            : 평균   5.0점 (최대 15점)
  트렌드 일치         : 평균   3.9점 (최대 10점)
  방향 일치도         : 평균  10.0점 (최대 10점)
  통계적 유의성        : 평균   7.7점 (최대 10점)
  자기상관           : 평균   7.8점 (최대 10점)
  이상치 비율         : 평균   8.7점 (최대 10점)
  정규성            : 평균   2.2점 (최대 5점)

최종 사용 쌍: 2757개

학습 데이터 생성
✓ 학습 데이터: (82710, 30)

모델 학습
학습 샘플: 82,710, 피처: 29

LGBM 학습 중...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overh

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
예측: 262it [00:01, 181.61it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warni


✓ 예측 완료: 2757개

🔎 follower 중복/불일치 요약 (후처리 전)
- 전체 row: 2,757
- follower 개수(유니크): 88
- follower당 평균 row 수: 31.33
- follower당 중앙값 row 수: 35
- value가 2개 이상인 follower 수: 87
  (상위 5개 follower row 수)
                    n  nunique
following_item_id             
WBLJNPZQ           57       27
XUOIQPFL           54       14
JPBRUTWP           52       14
OKMBFVKS           51       13
RAWUKQMJ           51       14

🔎 follower 중복/불일치 요약 (후처리 후 : clip)
- 전체 row: 2,757
- follower 개수(유니크): 88
- follower당 평균 row 수: 31.33
- follower당 중앙값 row 수: 35
- value가 2개 이상인 follower 수: 87
  (상위 5개 follower row 수)
                    n  nunique
following_item_id             
WBLJNPZQ           57       26
XUOIQPFL           54       12
JPBRUTWP           52       13
OKMBFVKS           51       13
RAWUKQMJ           51       14

✅ 후처리 적용 결과 체크
- 바뀐 row 수: 77 / 2,757 (2.79%)
- |log 변화| 평균: 0.0001, 중앙값: 0.0000, 최대: 0.1192

(변화가 큰 상위 샘플)
     leading_item_id following_item_id  value_before  value_after  \
641    

# 옵션2 - 그래프 기반 정보 추가 학습 코드 (이 정보를 모델링에 넣을 FEATURE로 쓰기) <- 피처로 추가하니 성능이 떨어짐

In [ ]:
# import datetime
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LinearRegression
# from lightgbm import LGBMRegressor
# from tqdm import tqdm

# train = pd.read_csv("train.csv")

# # --- 데이터 전처리 (변경 없음) ---
# monthly = (
#     train
#     .groupby(["item_id", "year", "month"], as_index=False)["value"]
#     .sum()
# )
# monthly["ym"] = pd.to_datetime(
#     monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
# )
# pivot = (
#     monthly
#     .pivot(index="item_id", columns="ym", values="value")
#     .fillna(0.0)
# )
# months_dt = pivot.columns.to_list()  # 월(datetime) 리스트

# # 공행성쌍 (DTW 포함)
# print("--- '원본 값' 기반 공행성 쌍 탐색 시작 ---")
# pairs = pairs_with_dtw
# print("탐색된 공행성쌍 수:", len(pairs))


# # ============================================================
# # (2) 학습 데이터 구축 — 여기서 “그래프 피처 3개만” 사용
# # ============================================================
# def build_training_data_optimized(pivot, pairs, months_dt):
#     months = months_dt
#     n_months = len(months)
#     rows = []

#     for row in pairs.itertuples(index=False):
#         leader = row.leading_item_id
#         follower = row.following_item_id
#         lag = int(row.best_lag)
#         corr = float(row.max_corr)

#         # ===== 그래프 기반 최종 선택된 3개 feature =====
#         same_comm  = float(row.same_comm)
#         deg_ratio  = float(row.deg_ratio)
#         btwn_ratio = float(row.btwn_ratio)

#         if leader not in pivot.index or follower not in pivot.index:
#             continue

#         a_series = pivot.loc[leader].values.astype(float)
#         b_series = pivot.loc[follower].values.astype(float)

#         for t in range(max(lag, 12), n_months - 1):
#             b_t = b_series[t]
#             b_t_1 = b_series[t - 1]
#             b_t_2 = b_series[t - 2]
#             b_t_12 = b_series[t - 12]
#             a_t_lag = a_series[t - lag]
#             b_t_plus_1 = b_series[t + 1]

#             target_month = months[t + 1].month

#             # 기존 피처 원형 유지
#             a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
#             a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
#             a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

#             b_diff = b_t - b_t_1
#             b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
#             b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
#             b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
#             b_roll_mean_6 = b_series[t-5:t+1].mean()
#             b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

#             ab_ratio = a_t_lag / (b_t + 1)
#             ab_diff = a_t_lag - b_t
#             corr_weighted_a = a_t_lag * corr

#             b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
#             b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
#             b_yoy_ratio = b_t / (b_t_12 + 1)
#             b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
#             recent_12 = b_series[max(0, t-11):t+1]
#             b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

#             quarter = (target_month - 1) // 3 + 1
#             is_year_end = 1 if target_month in [11, 12, 1] else 0
#             lag_weight = 1 / (1 + lag)

#             rows.append({
#                 # 기존 28개 피처
#                 "b_t": b_t,
#                 "b_t_1": b_t_1,
#                 "b_t_12": b_t_12,
#                 "a_t_lag": a_t_lag,
#                 "max_corr": corr,
#                 "best_lag": float(lag),
#                 "month": float(target_month),

#                 "a_t_lag_diff": a_t_lag_diff,
#                 "a_momentum": a_momentum,
#                 "a_roll_mean_3": a_roll_mean_3,

#                 "b_diff": b_diff,
#                 "b_pct_change": b_pct_change,
#                 "b_yoy_growth": b_yoy_growth,
#                 "b_roll_mean_3": b_roll_mean_3,
#                 "b_roll_mean_6": b_roll_mean_6,
#                 "b_roll_std_3": b_roll_std_3,

#                 "ab_ratio": ab_ratio,
#                 "ab_diff": ab_diff,
#                 "corr_weighted_a": corr_weighted_a,

#                 "b_cv": b_cv,
#                 "b_acceleration": b_acceleration,
#                 "b_yoy_ratio": b_yoy_ratio,
#                 "b_ma_ratio": b_ma_ratio,
#                 "b_percentile_rank": b_percentile_rank,

#                 "quarter": float(quarter),
#                 "is_year_end": is_year_end,

#                 "lag_weight": lag_weight,

#                 # ===== 추가된 그래프 기반 피처 3개 =====
#                 "same_comm": same_comm,
#                 "deg_ratio": deg_ratio,
#                 "btwn_ratio": btwn_ratio,

#                 "target": b_t_plus_1,
#             })

#     return pd.DataFrame(rows)


# df_train_model = build_training_data_optimized(pivot, pairs, months_dt)
# print("생성된 학습 데이터 shape:", df_train_model.shape)


# # ============================================================
# # (3) Feature 리스트 — 총 31개 (28 + 3)
# # ============================================================
# feature_cols = [
#     'b_t','b_t_1','b_t_12','a_t_lag','max_corr','best_lag','month',
#     'a_t_lag_diff','a_momentum','a_roll_mean_3',
#     'b_diff','b_pct_change','b_yoy_growth',
#     'b_roll_mean_3','b_roll_mean_6','b_roll_std_3',
#     'ab_ratio','ab_diff','corr_weighted_a',
#     'b_cv','b_acceleration','b_yoy_ratio','b_ma_ratio','b_percentile_rank',
#     'quarter','is_year_end',
#     'lag_weight',

#     # 그래프 기반 핵심 3개
#     'same_comm', 'deg_ratio', 'btwn_ratio'
# ]


# # ============================================================
# # (4) 모델 학습
# # ============================================================
# if df_train_model.empty:
#     print("학습 데이터가 비어있습니다.")
#     submission = pd.DataFrame(columns=['leading_item_id','following_item_id','value'])
# else:
#     train_X = df_train_model[feature_cols].values
#     train_y = df_train_model["target"].values

#     reg = LGBMRegressor(
#         random_state=42,
#         n_estimators=700,
#         learning_rate=0.01,
#         max_depth=8,
#         num_leaves=50,
#         min_child_samples=20,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_alpha=0.1,
#         reg_lambda=0.1,
#         min_split_gain=0.01
#     )
#     print("LGBM 학습 시작...")
#     print(f"총 피처: {len(feature_cols)}개")
#     reg.fit(train_X, train_y)
#     print("모델 학습 완료.")


# # ============================================================
# # (5) 예측 코드 — 동일한 3개 그래프 피처 추가
# # ============================================================
# def predict_optimized(pivot, pairs, reg, months_dt):
#     months = months_dt
#     n_months = len(months)
#     t_last = n_months - 1
#     t_prev = n_months - 2
#     preds = []

#     target_month = months[-1].month + 1 if months[-1].month < 12 else 1

#     for row in tqdm(pairs.itertuples(index=False)):
#         leader = row.leading_item_id
#         follower = row.following_item_id
#         lag = int(row.best_lag)
#         corr = float(row.max_corr)

#         # 그래프 기반 3개
#         same_comm  = float(row.same_comm)
#         deg_ratio  = float(row.deg_ratio)
#         btwn_ratio = float(row.btwn_ratio)

#         if leader not in pivot.index or follower not in pivot.index:
#             continue

#         a_series = pivot.loc[leader].values.astype(float)
#         b_series = pivot.loc[follower].values.astype(float)

#         if t_last - lag < 0 or t_last < 12:
#             continue

#         b_t = b_series[t_last]
#         b_t_1 = b_series[t_prev]
#         b_t_2 = b_series[t_prev - 1]
#         b_t_12 = b_series[t_last - 12]
#         a_t_lag = a_series[t_last - lag]

#         # 기존 피처와 동일
#         a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
#         a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
#         a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

#         b_diff = b_t - b_t_1
#         b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
#         b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
#         b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
#         b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
#         b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

#         ab_ratio = a_t_lag / (b_t + 1)
#         ab_diff = a_t_lag - b_t
#         corr_weighted_a = a_t_lag * corr

#         b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
#         b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
#         b_yoy_ratio = b_t / (b_t_12 + 1)
#         b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
#         recent_12 = b_series[max(0, t_last-11):t_last+1]
#         b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

#         quarter = (target_month - 1) // 3 + 1
#         is_year_end = 1 if target_month in [11,12,1] else 0

#         lag_weight = 1 / (1 + lag)

#         X_test = np.array([[
#             b_t,b_t_1,b_t_12,a_t_lag,corr,float(lag),float(target_month),
#             a_t_lag_diff,a_momentum,a_roll_mean_3,
#             b_diff,b_pct_change,b_yoy_growth,b_roll_mean_3,b_roll_mean_6,b_roll_std_3,
#             ab_ratio,ab_diff,corr_weighted_a,
#             b_cv,b_acceleration,b_yoy_ratio,b_ma_ratio,b_percentile_rank,
#             float(quarter),is_year_end,
#             lag_weight,
#             same_comm,deg_ratio,btwn_ratio
#         ]])

#         y_pred = reg.predict(X_test)[0]
#         y_pred = max(0.0, float(y_pred))
#         y_pred = int(round(y_pred))

#         preds.append({
#             "leading_item_id": leader,
#             "following_item_id": follower,
#             "value": y_pred,
#         })

#     return pd.DataFrame(preds)



# if df_train_model.empty:
#     print("학습된 모델 없음 → 예측 skip")
# else:
#     print("--- 예측 시작 ---")
#     submission = predict_optimized(pivot, pairs, reg, months_dt)
#     print(submission.head())

# date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# file_name = f"./result/hs_advanced_{date_str}.csv"
# submission.to_csv(file_name, index=False)

# print(f"✓ 파일 저장 완료: {file_name}")
# print("LGBM (고급 + 그래프 feature 3개) 예측 완료.")

1. 메타 모델 스태킹 (가장 강력 추천) ⭐⭐⭐
현재 0.7:0.3 하드코딩된 가중치는 데이터 기반 최적화가 안 된 상태입니다. 메타 모델을 사용하면:

왜 효과적인가: 2025년 1-7월 실제 정답 데이터가 있으므로, 이 기간에서 LGBM과 XGB의 최적 조합을 학습 가능
리스크: 매우 낮음 (최악의 경우 현재와 비슷한 성능)
구현 난이도: 낮음
Ridge로 메타 모델 스태킹해보기


In [ ]:
"""
무역 데이터 공행성 예측 모델 (2,700개 목표 최적화 + Ridge 메타 스태킹)
- 원본: 3,368개
- 목표: 2,700개 (약 80% 유지)
- 적절한 노이즈 제거 (중간 모드)
- 32개 피처 + LGBM/XGB + Ridge 메타 모델
"""

import datetime
import pandas as pd
import numpy as np
from scipy import stats
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from tqdm import tqdm

# ============================================================================
# 데이터 로드
# ============================================================================

print("="*80)
print("데이터 로드")
print("="*80)

monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({'value': 'sum', 'weight': 'sum'})
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

pivot = monthly.pivot(index="item_id", columns="ym", values="value").fillna(0.0)
pivot_weight = monthly.pivot(index="item_id", columns="ym", values="weight").fillna(0.0)
months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료: {len(pivot)}개 품목, {len(months_dt)}개월")

# ============================================================================
# ⭐ 중간 품질 점수 계산 (2,700개 목표용)
# ============================================================================

def calculate_quality_score_moderate(a_series, b_series, lag):
    """
    중간 품질 점수 계산 (0-100점)
    - 원본 3,368개 → 목표 2,700개
    - 적절한 노이즈 제거 (80% 유지)
    """

    scores = {}

    # 1. 0 비율 (10점) - 중간
    zero_ratio_a = (a_series == 0).sum() / len(a_series)
    zero_ratio_b = (b_series == 0).sum() / len(b_series)
    max_zero = max(zero_ratio_a, zero_ratio_b)

    if max_zero < 0.17:
        scores['zero'] = 10
    elif max_zero < 0.37:
        scores['zero'] = 7
    elif max_zero < 0.57:
        scores['zero'] = 4
    else:
        scores['zero'] = 0

    # 2. 변동계수 (10점) - 중간
    cv_a = np.std(a_series) / (np.mean(a_series) + 1)
    cv_b = np.std(b_series) / (np.mean(b_series) + 1)
    max_cv = max(cv_a, cv_b)

    if max_cv < 0.75:
        scores['cv'] = 10
    elif max_cv < 1.4:
        scores['cv'] = 7
    elif max_cv < 2.7:
        scores['cv'] = 4
    else:
        scores['cv'] = 0

    # 3. 스파이크 (10점) - 중간
    a_changes = np.abs(np.diff(a_series))
    b_changes = np.abs(np.diff(b_series))

    if len(a_changes) > 0 and len(b_changes) > 0:
        a_spike = a_changes.max() / (np.median(a_changes) + 1)
        b_spike = b_changes.max() / (np.median(b_changes) + 1)
        max_spike = max(a_spike, b_spike)

        if max_spike < 4.5:
            scores['spike'] = 10
        elif max_spike < 9.5:
            scores['spike'] = 7
        elif max_spike < 17:
            scores['spike'] = 4
        else:
            scores['spike'] = 0
    else:
        scores['spike'] = 5

    # 4. 안정성 (15점) - 중간
    n = len(a_series)

    try:
        mid = n // 2

        if mid >= 12 and mid + lag < len(b_series):
            corr_first = np.corrcoef(
                a_series[:mid],
                b_series[lag:mid+lag]
            )[0, 1]

            corr_second = np.corrcoef(
                a_series[mid:],
                b_series[mid+lag:]
            )[0, 1]

            if not np.isnan(corr_first) and not np.isnan(corr_second):
                consistency = 1 - abs(corr_first - corr_second)

                if consistency > 0.52:
                    scores['stability'] = 15
                elif consistency > 0.35:
                    scores['stability'] = 10
                elif consistency > 0.22:
                    scores['stability'] = 6
                else:
                    scores['stability'] = 3
            else:
                scores['stability'] = 5
        else:
            scores['stability'] = 5
    except:
        scores['stability'] = 5

    # 5. 트렌드 일치 (10점) - 중간
    if len(a_series) > 0 and len(b_series) > 0:
        x = np.arange(len(a_series))

        if np.std(a_series) > 0:
            slope_a, _ = np.polyfit(x, a_series, 1)
        else:
            slope_a = 0

        if np.std(b_series) > 0:
            slope_b, _ = np.polyfit(x, b_series, 1)
        else:
            slope_b = 0

        if (slope_a > 0 and slope_b > 0) or (slope_a < 0 and slope_b < 0):
            ratio = abs(slope_a) / (abs(slope_b) + 1e-10)
            if 0.38 < ratio < 2.7:
                scores['trend'] = 10
            elif 0.25 < ratio < 4.2:
                scores['trend'] = 6
            else:
                scores['trend'] = 3
        elif slope_a == 0 or slope_b == 0:
            scores['trend'] = 5
        else:
            scores['trend'] = 2
    else:
        scores['trend'] = 4

    # 6. 방향 일치도 (10점) - 중간
    if len(a_changes) > lag and len(b_changes) > lag:
        direction_match = (np.sign(a_changes[:-lag]) == np.sign(b_changes[lag:])).mean()

        if direction_match > 0.64:
            scores['direction'] = 10
        elif direction_match > 0.54:
            scores['direction'] = 7
        elif direction_match > 0.44:
            scores['direction'] = 4
        else:
            scores['direction'] = 1
    else:
        scores['direction'] = 4

    # 7. 통계적 유의성 (10점) - 중간
    try:
        if len(a_series) > lag:
            _, p_value = stats.pearsonr(
                a_series[:-lag] if lag > 0 else a_series,
                b_series[lag:] if lag > 0 else b_series
            )

            if p_value < 0.007:
                scores['significance'] = 10
            elif p_value < 0.025:
                scores['significance'] = 7
            elif p_value < 0.07:
                scores['significance'] = 4
            else:
                scores['significance'] = 1
        else:
            scores['significance'] = 4
    except:
        scores['significance'] = 4

    # 8. 자기상관 (10점) - 중간
    try:
        if len(b_series) > 1:
            autocorr = np.corrcoef(b_series[:-1], b_series[1:])[0, 1]

            if 0.24 < autocorr < 0.86:
                scores['autocorr'] = 10
            elif 0.09 < autocorr < 0.91:
                scores['autocorr'] = 7
            else:
                scores['autocorr'] = 3
        else:
            scores['autocorr'] = 4
    except:
        scores['autocorr'] = 4

    # 9. 이상치 비율 (10점) - 중간
    def count_outliers(series):
        q1 = np.percentile(series, 25)
        q3 = np.percentile(series, 75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        return outliers / len(series)

    outlier_ratio_a = count_outliers(a_series)
    outlier_ratio_b = count_outliers(b_series)
    max_outlier = max(outlier_ratio_a, outlier_ratio_b)

    if max_outlier < 0.085:
        scores['outlier'] = 10
    elif max_outlier < 0.16:
        scores['outlier'] = 7
    elif max_outlier < 0.26:
        scores['outlier'] = 4
    else:
        scores['outlier'] = 1

    # 10. 분포 정규성 (5점) - 중간
    try:
        if len(b_series) >= 8:
            _, p_shapiro = stats.shapiro(b_series)

            if p_shapiro > 0.02:
                scores['normality'] = 5
            elif p_shapiro > 0.004:
                scores['normality'] = 3
            else:
                scores['normality'] = 1
        else:
            scores['normality'] = 2
    except:
        scores['normality'] = 2

    total_score = sum(scores.values())

    return total_score, scores


# ============================================================================
# ⭐ 2,700개 목표 노이즈 제거
# ============================================================================

def remove_noise_pairs_for_2700(pairs, pivot, target_count=2700):
    """
    2,700개 목표 노이즈 제거

    원본: 3,368개
    목표: 2,700개 (약 80% 유지)
    전략: 적절한 노이즈 제거 (중간 모드)
    """

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 노이즈 제거 (중간 모드)")
    print("="*80)
    print(f"원본: {len(pairs):,}개")
    print(f"목표: {target_count:,}개 (약 {target_count/len(pairs)*100:.0f}% 유지)")

    quality_scores = []
    detailed_scores = []

    print(f"\n품질 평가 중...")
    for row in tqdm(pairs.itertuples(), total=len(pairs), desc="평가"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            quality_scores.append(0)
            detailed_scores.append({})
            continue

        a_series = pivot.loc[leader].values
        b_series = pivot.loc[follower].values

        total_score, scores = calculate_quality_score_moderate(a_series, b_series, lag)

        quality_scores.append(total_score)
        detailed_scores.append(scores)

    pairs_with_score = pairs.copy()
    pairs_with_score['quality_score'] = quality_scores

    sorted_scores = sorted(quality_scores, reverse=True)

    if len(sorted_scores) >= target_count:
        threshold = sorted_scores[target_count - 1]
        threshold = max(33, (threshold // 5) * 5)
    else:
        threshold = 33

    pairs_clean = pairs_with_score[
        pairs_with_score['quality_score'] >= threshold
    ].copy()

    pairs_for_model = pairs_clean.drop(columns=['quality_score'], errors='ignore')

    removed = len(pairs) - len(pairs_clean)
    removed_ratio = removed / len(pairs) * 100
    kept_ratio = len(pairs_clean) / len(pairs) * 100

    print(f"\n{'='*80}")
    print(f"✅ 노이즈 제거 완료!")
    print(f"{'='*80}")
    print(f"원본: {len(pairs):,}개")
    print(f"제거: {removed:,}개 ({removed_ratio:.1f}%)")
    print(f"유지: {len(pairs_clean):,}개 ({kept_ratio:.1f}%)")
    print(f"임계값: {threshold}점")

    return pairs_for_model


# ============================================================================
# 조건부 Weight 피처
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """조건부 Weight 피처 4개"""
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    weight_value_ratio = weight * (value / (value_median + 1))
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    if value > value_q75:
        weight_importance = weight * 1.5
    elif value > value_median:
        weight_importance = weight * 1.0
    else:
        weight_importance = weight * 0.1

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }


# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt, exclude_recent_months=0):
    """
    32개 피처 학습 데이터 생성

    exclude_recent_months: 최근 N개월 제외 (메타 학습용)
    """
    months = months_dt
    n_months = len(months)

    # 메타 학습용 데이터 제외
    max_t = n_months - 1 - exclude_recent_months

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        for t in range(max(lag, 12), max_t):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]
            b_weight_t = b_weight_series[t]
            target_month = months[t + 1].month

            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0
            lag_weight = 1 / (1 + lag)

            b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

            rows.append({
                "b_t": b_t, "b_t_1": b_t_1, "b_t_12": b_t_12, "a_t_lag": a_t_lag,
                "max_corr": corr, "best_lag": float(lag), "month": float(target_month),
                "a_t_lag_diff": a_t_lag_diff, "a_momentum": a_momentum, "a_roll_mean_3": a_roll_mean_3,
                "b_diff": b_diff, "b_pct_change": b_pct_change, "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3, "b_roll_mean_6": b_roll_mean_6, "b_roll_std_3": b_roll_std_3,
                "ab_ratio": ab_ratio, "ab_diff": ab_diff, "corr_weighted_a": corr_weighted_a,
                "b_cv": b_cv, "b_acceleration": b_acceleration, "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio, "b_percentile_rank": b_percentile_rank,
                "quarter": float(quarter), "is_year_end": is_year_end, "lag_weight": lag_weight,
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                "target": b_t_plus_1,
            })

    return pd.DataFrame(rows)


# ============================================================================
# ⭐ 메타 학습 데이터 생성 (2025년 1-7월)
# ============================================================================

def build_meta_training_data(pivot, pivot_weight, pairs, months_dt, model_lgb, model_xgb,
                              feature_cols, meta_months=7):
    """
    2025년 최근 N개월 데이터로 메타 모델 학습용 데이터 생성

    Returns:
        meta_X: LGBM, XGB 예측값 (N, 2)
        meta_y: 실제 정답값 (N,)
    """
    months = months_dt
    n_months = len(months)

    # 메타 학습 시작 인덱스 (최근 meta_months+1개월)
    meta_start_idx = n_months - meta_months - 1

    meta_preds_lgb = []
    meta_preds_xgb = []
    meta_targets = []

    print(f"\n{'='*80}")
    print(f"⭐ 메타 학습 데이터 생성 (최근 {meta_months}개월)")
    print(f"{'='*80}")

    for row in tqdm(pairs.itertuples(index=False), desc="메타 데이터"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        # 최근 meta_months개월에 대해 예측
        for t in range(meta_start_idx, n_months - 1):
            if t - lag < 0 or t < 12:
                continue

            # 피처 생성 (기존과 동일)
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_weight_t = b_weight_series[t]
            target_month = months[t + 1].month

            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = b_series[t-5:t+1].mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0
            lag_weight = 1 / (1 + lag)

            b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

            # 피처 배열 생성
            X_features = [
                b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
                a_t_lag_diff, a_momentum, a_roll_mean_3,
                b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
                ab_ratio, ab_diff, corr_weighted_a,
                b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
                float(quarter), is_year_end, lag_weight,
                b_weight_features['weight_for_high_value'],
                b_weight_features['weight_value_ratio'],
                b_weight_features['exp_weighted'],
                b_weight_features['weight_importance']
            ]

            X_test = np.array([X_features])

            # 두 모델의 예측값
            pred_lgb = model_lgb.predict(X_test)[0]
            pred_xgb = model_xgb.predict(X_test)[0]

            # 실제 정답값
            actual = b_series[t + 1]

            meta_preds_lgb.append(pred_lgb)
            meta_preds_xgb.append(pred_xgb)
            meta_targets.append(actual)

    # (N, 2) 형태로 변환
    meta_X = np.column_stack([meta_preds_lgb, meta_preds_xgb])
    meta_y = np.array(meta_targets)

    print(f"✓ 메타 학습 샘플: {len(meta_y):,}개")

    return meta_X, meta_y


# ============================================================================
# ⭐ Ridge 메타 모델 학습
# ============================================================================

def train_meta_model(meta_X, meta_y):
    """
    Ridge 회귀로 메타 모델 학습

    alpha=1.0: L2 regularization 강도
    fit_intercept=False: 절편 없음 (순수 가중치만)
    positive=True: 음수 가중치 방지
    """
    print(f"\n{'='*80}")
    print(f"⭐ Ridge 메타 모델 학습")
    print(f"{'='*80}")

    meta_model = Ridge(alpha=1.0, fit_intercept=False, positive=True)
    meta_model.fit(meta_X, meta_y)

    # 학습된 가중치
    w_lgb, w_xgb = meta_model.coef_
    total_weight = w_lgb + w_xgb
    w_lgb_norm = w_lgb / total_weight
    w_xgb_norm = w_xgb / total_weight

    print(f"\n✓ Ridge 메타 모델 학습 완료!")
    print(f"{'='*80}")
    print(f"학습된 가중치:")
    print(f"  - LGBM: {w_lgb_norm:.4f} (기존: 0.7000)")
    print(f"  - XGB:  {w_xgb_norm:.4f} (기존: 0.3000)")
    print(f"{'='*80}")

    # 메타 학습 데이터에서의 성능 확인
    meta_pred = meta_model.predict(meta_X)
    mae = np.mean(np.abs(meta_y - meta_pred))
    rmse = np.sqrt(np.mean((meta_y - meta_pred) ** 2))

    print(f"\n메타 학습 데이터 성능:")
    print(f"  - MAE:  {mae:,.2f}")
    print(f"  - RMSE: {rmse:,.2f}")

    # 기존 0.7:0.3 가중치와 비교
    baseline_pred = 0.7 * meta_X[:, 0] + 0.3 * meta_X[:, 1]
    baseline_mae = np.mean(np.abs(meta_y - baseline_pred))
    baseline_rmse = np.sqrt(np.mean((meta_y - baseline_pred) ** 2))

    print(f"\n기존 0.7:0.3 가중치 성능:")
    print(f"  - MAE:  {baseline_mae:,.2f} (차이: {mae - baseline_mae:+.2f})")
    print(f"  - RMSE: {baseline_rmse:,.2f} (차이: {rmse - baseline_rmse:+.2f})")

    return meta_model, (w_lgb_norm, w_xgb_norm)


# ============================================================================
# 예측 함수 (Ridge 메타 모델 사용)
# ============================================================================

def predict_with_meta_model(pivot, pivot_weight, pairs, model_lgb, model_xgb,
                             meta_model, months_dt):
    """
    Ridge 메타 모델을 사용한 최종 예측
    """
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []
    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    # 디버깅 카운터
    total_pairs = 0
    skipped_not_in_pivot = 0
    skipped_condition = 0
    success_count = 0

    print(f"\n{'='*80}")
    print(f"⭐ Ridge 메타 모델 예측")
    print(f"{'='*80}")
    print(f"전체 pairs: {len(pairs)}개")
    print(f"예측 시점: t_last={t_last}, n_months={n_months}")

    for row in tqdm(pairs.itertuples(index=False), desc="예측"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag < 0 or t_last < 12:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]
        b_weight_t = b_weight_series[t_last]

        a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
        a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0
        lag_weight = 1 / (1 + lag)

        b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

        X_features = [
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_t_lag_diff, a_momentum, a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end, lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance']
        ]

        X_test = np.array([X_features])

        # LGBM, XGB 예측
        pred_lgb = model_lgb.predict(X_test)[0]
        pred_xgb = model_xgb.predict(X_test)[0]

        # Ridge 메타 모델로 최종 예측
        meta_input = np.array([[pred_lgb, pred_xgb]])
        y_pred = meta_model.predict(meta_input)[0]

        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


# ============================================================================
# 메인 실행
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("⭐ Ridge 메타 스태킹 버전 (2,700개 목표)")
    print("="*80)

    # ⭐ 2,700개 목표 노이즈 제거
    pairs_for_model = remove_noise_pairs_for_2700(
        pairs=pairs_with_dtw,
        pivot=pivot,
        target_count=2700
    )

    print(f"\n최종 사용 쌍: {len(pairs_for_model)}개")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_t_lag_diff', 'a_momentum', 'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth', 'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end', 'lag_weight',
        'b_weight_for_high_value', 'b_weight_value_ratio', 'b_exp_weighted', 'b_weight_importance'
    ]

    # ============================================================================
    # 1단계: Base 모델 학습 (2022~2024 데이터, 최근 7개월 제외)
    # ============================================================================

    print("\n" + "="*80)
    print("1단계: Base 모델 학습 (LGBM, XGBoost)")
    print("="*80)
    print("학습 데이터: 2022~2024 + 2025년 초반 (최근 7개월 제외)")

    # 최근 7개월을 제외한 학습 데이터 생성
    df_train_base = build_training_data_optimized(
        pivot, pivot_weight, pairs_for_model, months_dt,
        exclude_recent_months=7  # 최근 7개월 제외
    )

    print(f"✓ Base 학습 데이터: {df_train_base.shape}")

    if not df_train_base.empty:
        train_X = df_train_base[feature_cols].values
        train_y = df_train_base["target"].values

        print(f"Base 학습 샘플: {len(train_X):,}, 피처: {len(feature_cols)}")

        # LGBM 학습
        model_lgb = LGBMRegressor(
            random_state=42, n_estimators=700, learning_rate=0.01, max_depth=8,
            num_leaves=50, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, min_split_gain=0.01, verbose=-1
        )

        print("\nLGBM 학습 중...")
        model_lgb.fit(train_X, train_y)
        print("✓ LGBM 완료")

        # XGBoost 학습
        model_xgb = XGBRegressor(
            random_state=42, n_estimators=900, learning_rate=0.02, max_depth=8,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
            tree_method='hist', verbosity=0
        )

        print("XGBoost 학습 중...")
        model_xgb.fit(train_X, train_y)
        print("✓ XGBoost 완료")

        # ============================================================================
        # 2단계: 메타 모델 학습 (2025년 최근 7개월)
        # ============================================================================

        # 메타 학습 데이터 생성
        meta_X, meta_y = build_meta_training_data(
            pivot, pivot_weight, pairs_for_model, months_dt,
            model_lgb, model_xgb, feature_cols, meta_months=7
        )

        # Ridge 메타 모델 학습
        meta_model, weights = train_meta_model(meta_X, meta_y)

        # ============================================================================
        # 3단계: 최종 예측 (Ridge 메타 모델 사용)
        # ============================================================================

        submission = predict_with_meta_model(
            pivot, pivot_weight, pairs_for_model,
            model_lgb, model_xgb, meta_model, months_dt
        )

        print(f"\n✓ 예측 완료: {len(submission)}개")

        # 저장
        date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        submission.to_csv('sy_2700_ridge_meta_{date_str}.csv', index=False)

        print(f"\n{'='*80}")

        print(f"{'='*80}")
        print("\n✅ Ridge 메타 스태킹 버전 완료!")
        print("="*80)
        print("주요 개선사항:")
        print("  ✓ 원본: 3,368개 → 목표: 2,700개")
        print("  ✓ 중간 기준 (적절한 노이즈 제거)")
        print("  ✓ 32개 피처")
        print(f"  ✓ LGBM + XGB + Ridge 메타 스태킹")
        print(f"  ✓ 학습된 가중치: LGBM {weights[0]:.4f}, XGB {weights[1]:.4f}")
        print("="*80)
    else:
        print("\n❌ 학습 데이터가 비어있습니다!")

데이터 로드
✓ 데이터 로드 완료: 100개 품목, 43개월

⭐ Ridge 메타 스태킹 버전 (2,700개 목표)

⭐ 2,700개 목표 노이즈 제거 (중간 모드)
원본: 3,368개
목표: 2,700개 (약 80% 유지)

품질 평가 중...


평가: 100%|██████████| 3368/3368 [00:07<00:00, 422.80it/s]



✅ 노이즈 제거 완료!
원본: 3,368개
제거: 611개 (18.1%)
유지: 2,757개 (81.9%)
임계값: 55점

최종 사용 쌍: 2757개

1단계: Base 모델 학습 (LGBM, XGBoost)
학습 데이터: 2022~2024 + 2025년 초반 (최근 7개월 제외)
✓ Base 학습 데이터: (63411, 32)
Base 학습 샘플: 63,411, 피처: 31

LGBM 학습 중...
✓ LGBM 완료
XGBoost 학습 중...
✓ XGBoost 완료

⭐ 메타 학습 데이터 생성 (최근 7개월)


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
메타 데이터: 2400it [04:27,  9.57it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  w

✓ 메타 학습 샘플: 19,299개

⭐ Ridge 메타 모델 학습

✓ Ridge 메타 모델 학습 완료!
학습된 가중치:
  - LGBM: 0.3792 (기존: 0.7000)
  - XGB:  0.6208 (기존: 0.3000)

메타 학습 데이터 성능:
  - MAE:  1,223,570.06
  - RMSE: 3,641,501.37

기존 0.7:0.3 가중치 성능:
  - MAE:  1,206,920.63 (차이: +16649.43)
  - RMSE: 3,681,145.10 (차이: -39643.73)

⭐ Ridge 메타 모델 예측
전체 pairs: 2757개
예측 시점: t_last=42, n_months=43


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
예측: 258it [00:04, 51.62it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnin


✓ 예측 완료: 2757개


✅ Ridge 메타 스태킹 버전 완료!
주요 개선사항:
  ✓ 원본: 3,368개 → 목표: 2,700개
  ✓ 중간 기준 (적절한 노이즈 제거)
  ✓ 32개 피처
  ✓ LGBM + XGB + Ridge 메타 스태킹
  ✓ 학습된 가중치: LGBM 0.3792, XGB 0.6208


#왜 Ridge 메타 스태킹이 성능이 떨어졌을까?
베이스 모델이 보는 양 차이
7개월간 보고 있기 때문에 만약 2025-01~07 구간에서 우연히 XGB가 좀 더 잘 맞았고 Ridge가 XGB 비중을 0.62로 높게줬는데 실제 08월에는 LBGM이 더 잘 맞는 구간이 많다면 0.7/0.3 고정이 오히려 더 보수적인 평균 역할을 하면서 잘 버티는 것



In [ ]:
"""
무역 데이터 공행성 예측 모델 (2,700개 목표 최적화)
- 원본: 3,368개
- 목표: 2,700개 (약 80% 유지)
- 적절한 노이즈 제거 (중간 모드)
- 32개 피처 + LGBM/XGB 앙상블
"""

import datetime
import pandas as pd
import numpy as np
from scipy import stats
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from tqdm import tqdm

# ============================================================================
# 데이터 로드
# ============================================================================

print("="*80)
print("데이터 로드")
print("="*80)

monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({'value': 'sum', 'weight': 'sum'})
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

pivot = monthly.pivot(index="item_id", columns="ym", values="value").fillna(0.0)
pivot_weight = monthly.pivot(index="item_id", columns="ym", values="weight").fillna(0.0)
months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료: {len(pivot)}개 품목, {len(months_dt)}개월")

# ============================================================================
# ⭐ 중간 품질 점수 계산 (2,700개 목표용)
# ============================================================================

def calculate_quality_score_moderate(a_series, b_series, lag):
    """
    중간 품질 점수 계산 (0-100점)
    - 원본 3,368개 → 목표 2,700개
    - 적절한 노이즈 제거 (80% 유지)
    """

    scores = {}

    # 1. 0 비율 (10점) - 중간
    zero_ratio_a = (a_series == 0).sum() / len(a_series)
    zero_ratio_b = (b_series == 0).sum() / len(b_series)
    max_zero = max(zero_ratio_a, zero_ratio_b)

    if max_zero < 0.17:
        scores['zero'] = 10
    elif max_zero < 0.37:
        scores['zero'] = 7
    elif max_zero < 0.57:  # 57%까지 허용
        scores['zero'] = 4
    else:
        scores['zero'] = 0

    # 2. 변동계수 (10점) - 중간
    cv_a = np.std(a_series) / (np.mean(a_series) + 1)
    cv_b = np.std(b_series) / (np.mean(b_series) + 1)
    max_cv = max(cv_a, cv_b)

    if max_cv < 0.75:
        scores['cv'] = 10
    elif max_cv < 1.4:
        scores['cv'] = 7
    elif max_cv < 2.7:  # 2.7까지 허용
        scores['cv'] = 4
    else:
        scores['cv'] = 0

    # 3. 스파이크 (10점) - 중간
    a_changes = np.abs(np.diff(a_series))
    b_changes = np.abs(np.diff(b_series))

    if len(a_changes) > 0 and len(b_changes) > 0:
        a_spike = a_changes.max() / (np.median(a_changes) + 1)
        b_spike = b_changes.max() / (np.median(b_changes) + 1)
        max_spike = max(a_spike, b_spike)

        if max_spike < 4.5:
            scores['spike'] = 10
        elif max_spike < 9.5:
            scores['spike'] = 7
        elif max_spike < 17:  # 17까지 허용
            scores['spike'] = 4
        else:
            scores['spike'] = 0
    else:
        scores['spike'] = 5

    # 4. 안정성 (15점) - 중간
    n = len(a_series)

    try:
        mid = n // 2

        if mid >= 12 and mid + lag < len(b_series):
            corr_first = np.corrcoef(
                a_series[:mid],
                b_series[lag:mid+lag]
            )[0, 1]

            corr_second = np.corrcoef(
                a_series[mid:],
                b_series[mid+lag:]
            )[0, 1]

            if not np.isnan(corr_first) and not np.isnan(corr_second):
                consistency = 1 - abs(corr_first - corr_second)

                if consistency > 0.52:  # 중간
                    scores['stability'] = 15
                elif consistency > 0.35:
                    scores['stability'] = 10
                elif consistency > 0.22:
                    scores['stability'] = 6
                else:
                    scores['stability'] = 3
            else:
                scores['stability'] = 5
        else:
            scores['stability'] = 5
    except:
        scores['stability'] = 5

    # 5. 트렌드 일치 (10점) - 중간
    if len(a_series) > 0 and len(b_series) > 0:
        x = np.arange(len(a_series))

        if np.std(a_series) > 0:
            slope_a, _ = np.polyfit(x, a_series, 1)
        else:
            slope_a = 0

        if np.std(b_series) > 0:
            slope_b, _ = np.polyfit(x, b_series, 1)
        else:
            slope_b = 0

        if (slope_a > 0 and slope_b > 0) or (slope_a < 0 and slope_b < 0):
            ratio = abs(slope_a) / (abs(slope_b) + 1e-10)
            if 0.38 < ratio < 2.7:  # 중간
                scores['trend'] = 10
            elif 0.25 < ratio < 4.2:
                scores['trend'] = 6
            else:
                scores['trend'] = 3
        elif slope_a == 0 or slope_b == 0:
            scores['trend'] = 5
        else:
            scores['trend'] = 2
    else:
        scores['trend'] = 4

    # 6. 방향 일치도 (10점) - 중간
    if len(a_changes) > lag and len(b_changes) > lag:
        direction_match = (np.sign(a_changes[:-lag]) == np.sign(b_changes[lag:])).mean()

        if direction_match > 0.64:  # 중간
            scores['direction'] = 10
        elif direction_match > 0.54:
            scores['direction'] = 7
        elif direction_match > 0.44:
            scores['direction'] = 4
        else:
            scores['direction'] = 1
    else:
        scores['direction'] = 4

    # 7. 통계적 유의성 (10점) - 중간
    try:
        if len(a_series) > lag:
            _, p_value = stats.pearsonr(
                a_series[:-lag] if lag > 0 else a_series,
                b_series[lag:] if lag > 0 else b_series
            )

            if p_value < 0.007:  # 중간
                scores['significance'] = 10
            elif p_value < 0.025:
                scores['significance'] = 7
            elif p_value < 0.07:
                scores['significance'] = 4
            else:
                scores['significance'] = 1
        else:
            scores['significance'] = 4
    except:
        scores['significance'] = 4

    # 8. 자기상관 (10점) - 중간
    try:
        if len(b_series) > 1:
            autocorr = np.corrcoef(b_series[:-1], b_series[1:])[0, 1]

            if 0.24 < autocorr < 0.86:  # 중간
                scores['autocorr'] = 10
            elif 0.09 < autocorr < 0.91:
                scores['autocorr'] = 7
            else:
                scores['autocorr'] = 3
        else:
            scores['autocorr'] = 4
    except:
        scores['autocorr'] = 4

    # 9. 이상치 비율 (10점) - 중간
    def count_outliers(series):
        q1 = np.percentile(series, 25)
        q3 = np.percentile(series, 75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        return outliers / len(series)

    outlier_ratio_a = count_outliers(a_series)
    outlier_ratio_b = count_outliers(b_series)
    max_outlier = max(outlier_ratio_a, outlier_ratio_b)

    if max_outlier < 0.085:
        scores['outlier'] = 10
    elif max_outlier < 0.16:
        scores['outlier'] = 7
    elif max_outlier < 0.26:  # 26%까지 허용
        scores['outlier'] = 4
    else:
        scores['outlier'] = 1

    # 10. 분포 정규성 (5점) - 중간
    try:
        if len(b_series) >= 8:
            _, p_shapiro = stats.shapiro(b_series)

            if p_shapiro > 0.02:  # 중간
                scores['normality'] = 5
            elif p_shapiro > 0.004:
                scores['normality'] = 3
            else:
                scores['normality'] = 1
        else:
            scores['normality'] = 2
    except:
        scores['normality'] = 2

    total_score = sum(scores.values())

    return total_score, scores


# ============================================================================
# ⭐ 2,700개 목표 노이즈 제거
# ============================================================================

def remove_noise_pairs_for_2700(pairs, pivot, target_count=2700):
    """
    2,700개 목표 노이즈 제거

    원본: 3,368개
    목표: 2,700개 (약 80% 유지)
    전략: 적절한 노이즈 제거 (중간 모드)
    """

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 노이즈 제거 (중간 모드)")
    print("="*80)
    print(f"원본: {len(pairs):,}개")
    print(f"목표: {target_count:,}개 (약 {target_count/len(pairs)*100:.0f}% 유지)")

    quality_scores = []
    detailed_scores = []

    # 품질 점수 계산 (중간 기준)
    print(f"\n품질 평가 중...")
    for row in tqdm(pairs.itertuples(), total=len(pairs), desc="평가"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            quality_scores.append(0)
            detailed_scores.append({})
            continue

        a_series = pivot.loc[leader].values
        b_series = pivot.loc[follower].values

        total_score, scores = calculate_quality_score_moderate(a_series, b_series, lag)

        quality_scores.append(total_score)
        detailed_scores.append(scores)

    # 품질 점수 추가
    pairs_with_score = pairs.copy()
    pairs_with_score['quality_score'] = quality_scores

    # 목표 개수 기반 임계값 계산
    sorted_scores = sorted(quality_scores, reverse=True)

    if len(sorted_scores) >= target_count:
        # 목표 개수번째 점수를 임계값으로
        threshold = sorted_scores[target_count - 1]
        # 5점 단위로 내림
        threshold = max(33, (threshold // 5) * 5)  # 최소 33점
    else:
        threshold = 33  # 최소 임계값

    # 필터링
    pairs_clean = pairs_with_score[
        pairs_with_score['quality_score'] >= threshold
    ].copy()

    # 품질 점수 컬럼 제거
    pairs_for_model = pairs_clean.drop(columns=['quality_score'], errors='ignore')

    # 통계
    removed = len(pairs) - len(pairs_clean)
    removed_ratio = removed / len(pairs) * 100
    kept_ratio = len(pairs_clean) / len(pairs) * 100

    print(f"\n{'='*80}")
    print(f"✅ 노이즈 제거 완료!")
    print(f"{'='*80}")
    print(f"원본: {len(pairs):,}개")
    print(f"제거: {removed:,}개 ({removed_ratio:.1f}%)")
    print(f"유지: {len(pairs_clean):,}개 ({kept_ratio:.1f}%)")
    print(f"임계값: {threshold}점")

    # 점수 분포
    print(f"\n품질 점수 분포:")
    print(f"  - 평균: {np.mean(quality_scores):.1f}점")
    print(f"  - 중앙값: {np.median(quality_scores):.1f}점")
    print(f"  - 최소: {np.min(quality_scores):.1f}점")
    print(f"  - 최대: {np.max(quality_scores):.1f}점")

    # 선택된 쌍의 점수
    selected_scores = pairs_clean['quality_score'].values
    print(f"\n선택된 쌍의 점수:")
    print(f"  - 평균: {np.mean(selected_scores):.1f}점")
    print(f"  - 최소: {np.min(selected_scores):.1f}점")

    # 구간별 분포
    print(f"\n점수 구간별 분포:")
    bins = [0, 30, 40, 50, 60, 70, 80, 90, 100]
    for i in range(len(bins)-1):
        total_count = sum((np.array(quality_scores) >= bins[i]) & (np.array(quality_scores) < bins[i+1]))
        selected_count = sum((selected_scores >= bins[i]) & (selected_scores < bins[i+1]))
        print(f"  {bins[i]:3d}-{bins[i+1]:3d}점: 전체 {total_count:4,}개, 선택 {selected_count:4,}개")

    # 기준별 평균 점수
    print(f"\n기준별 평균 점수 (선택된 쌍):")
    criteria_names = {
        'zero': '0 비율',
        'cv': '변동계수',
        'spike': '스파이크',
        'stability': '안정성',
        'trend': '트렌드 일치',
        'direction': '방향 일치도',
        'significance': '통계적 유의성',
        'autocorr': '자기상관',
        'outlier': '이상치 비율',
        'normality': '정규성'
    }

    selected_indices = pairs_with_score[pairs_with_score['quality_score'] >= threshold].index
    selected_detailed = [detailed_scores[i] for i in selected_indices if i < len(detailed_scores)]

    for key, name in criteria_names.items():
        scores = [s.get(key, 0) for s in selected_detailed if s]
        if scores:
            if key == 'stability':
                max_score = 15
            elif key == 'normality':
                max_score = 5
            else:
                max_score = 10

            print(f"  {name:15s}: 평균 {np.mean(scores):5.1f}점 (최대 {max_score}점)")

    return pairs_for_model


# ============================================================================
# 조건부 Weight 피처
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """조건부 Weight 피처 4개"""
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    weight_value_ratio = weight * (value / (value_median + 1))
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    if value > value_q75:
        weight_importance = weight * 1.5
    elif value > value_median:
        weight_importance = weight * 1.0
    else:
        weight_importance = weight * 0.1

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }


# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt):
    """32개 피처 학습 데이터 생성"""
    months = months_dt
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        for t in range(max(lag, 12), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]
            b_weight_t = b_weight_series[t]
            target_month = months[t + 1].month

            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0
            lag_weight = 1 / (1 + lag)

            b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

            rows.append({
                "b_t": b_t, "b_t_1": b_t_1, "b_t_12": b_t_12, "a_t_lag": a_t_lag,
                "max_corr": corr, "best_lag": float(lag), "month": float(target_month),
                "a_t_lag_diff": a_t_lag_diff, "a_momentum": a_momentum, "a_roll_mean_3": a_roll_mean_3,
                "b_diff": b_diff, "b_pct_change": b_pct_change, "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3, "b_roll_mean_6": b_roll_mean_6, "b_roll_std_3": b_roll_std_3,
                "ab_ratio": ab_ratio, "ab_diff": ab_diff, "corr_weighted_a": corr_weighted_a,
                "b_cv": b_cv, "b_acceleration": b_acceleration, "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio, "b_percentile_rank": b_percentile_rank,
                "quarter": float(quarter), "is_year_end": is_year_end, "lag_weight": lag_weight,
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                "target": np.log1p(b_t_plus_1),
            })

    return pd.DataFrame(rows)


# ============================================================================
# 예측 함수
# ============================================================================

def predict_optimized(pivot, pivot_weight, pairs, model_lgb, model_xgb, months_dt):
    """32개 피처 예측 + 앙상블"""
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []
    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    for row in tqdm(pairs.itertuples(index=False), desc="예측"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag < 0 or t_last < 12:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]
        b_weight_t = b_weight_series[t_last]

        a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
        a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0
        lag_weight = 1 / (1 + lag)

        b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

        X_test = np.array([[
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_t_lag_diff, a_momentum, a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end, lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance']
        ]])

        pred_lgb = model_lgb.predict(X_test)[0]
        pred_xgb = model_xgb.predict(X_test)[0]
        y_pred_log = 0.7 * pred_lgb + 0.3 * pred_xgb
        y_pred = np.expm1(y_pred_log)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


# ============================================================================
# 메인 실행
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 최적화 버전")
    print("="*80)

    # ⭐ 2,700개 목표 노이즈 제거
    pairs_for_model = remove_noise_pairs_for_2700(
        pairs=pairs_with_dtw,
        pivot=pivot,
        target_count=2700      # ⭐ 목표: 2,700개
    )

    print(f"\n최종 사용 쌍: {len(pairs_for_model)}개")

    # 학습
    print("\n" + "="*80)
    print("학습 데이터 생성")
    print("="*80)

    df_train_model = build_training_data_optimized(pivot, pivot_weight, pairs_for_model, months_dt)
    print(f"✓ 학습 데이터: {df_train_model.shape}")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_t_lag_diff', 'a_momentum', 'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth', 'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end', 'lag_weight',
        'b_weight_for_high_value', 'b_weight_value_ratio', 'b_exp_weighted', 'b_weight_importance'
    ]

    if not df_train_model.empty:
        print("\n" + "="*80)
        print("모델 학습")
        print("="*80)

        train_X = df_train_model[feature_cols].values
        train_y = df_train_model["target"].values

        print(f"학습 샘플: {len(train_X):,}, 피처: {len(feature_cols)}")

        model_lgb = LGBMRegressor(
            random_state=42, n_estimators=700, learning_rate=0.01, max_depth=8,
            num_leaves=50, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, min_split_gain=0.01
        )

        print("\nLGBM 학습 중...")
        model_lgb.fit(train_X, train_y)
        print("✓ LGBM 완료")

        model_xgb = XGBRegressor(
            random_state=42, n_estimators=900, learning_rate=0.02, max_depth=8,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
            tree_method='hist'
        )

        print("XGBoost 학습 중...")
        model_xgb.fit(train_X, train_y)
        print("✓ XGBoost 완료")

        print("\n" + "="*80)
        print("예측")
        print("="*80)

        submission = predict_optimized(pivot, pivot_weight, pairs_for_model, model_lgb, model_xgb, months_dt)
        print(f"\n✓ 예측 완료: {len(submission)}개")

        date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        submission.to_csv(f'_result_sy_2700_moderate_{date_str}.csv', index=False)


        print("\n✅ 2,700개 목표 버전 완료!")
        print("  - 원본: 3,368개")
        print("  - 목표: 2,700개")
        print("  - 중간 기준 (적절한 노이즈 제거)")
        print("  - 32개 피처")
        print("  - LGBM + XGB 앙상블")
        print("="*80)

이건 seed를 앙상블 해봤는데 오히려 성능이 조금 떨어졌어요

In [ ]:
"""
무역 데이터 공행성 예측 모델 (2,700개 목표 최적화)
- 원본: 3,368개
- 목표: 2,700개 (약 80% 유지)
- 적절한 노이즈 제거 (중간 모드)
- 32개 피처 + LGBM/XGB 앙상블 (+ seed 앙상블)
"""

import datetime
import pandas as pd
import numpy as np
from scipy import stats
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from tqdm import tqdm

# ============================================================================
# 데이터 로드
# ============================================================================

print("="*80)
print("데이터 로드")
print("="*80)

monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)
    .agg({'value': 'sum', 'weight': 'sum'})
)

monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

pivot = monthly.pivot(index="item_id", columns="ym", values="value").fillna(0.0)
pivot_weight = monthly.pivot(index="item_id", columns="ym", values="weight").fillna(0.0)
months_dt = pivot.columns.to_list()

print(f"✓ 데이터 로드 완료: {len(pivot)}개 품목, {len(months_dt)}개월")

# ============================================================================
# ⭐ 중간 품질 점수 계산 (2,700개 목표용)
# ============================================================================

def calculate_quality_score_moderate(a_series, b_series, lag):
    """
    중간 품질 점수 계산 (0-100점)
    - 원본 3,368개 → 목표 2,700개
    - 적절한 노이즈 제거 (80% 유지)
    """

    scores = {}

    # 1. 0 비율 (10점) - 중간
    zero_ratio_a = (a_series == 0).sum() / len(a_series)
    zero_ratio_b = (b_series == 0).sum() / len(b_series)
    max_zero = max(zero_ratio_a, zero_ratio_b)

    if max_zero < 0.17:
        scores['zero'] = 10
    elif max_zero < 0.37:
        scores['zero'] = 7
    elif max_zero < 0.57:  # 57%까지 허용
        scores['zero'] = 4
    else:
        scores['zero'] = 0

    # 2. 변동계수 (10점) - 중간
    cv_a = np.std(a_series) / (np.mean(a_series) + 1)
    cv_b = np.std(b_series) / (np.mean(b_series) + 1)
    max_cv = max(cv_a, cv_b)

    if max_cv < 0.75:
        scores['cv'] = 10
    elif max_cv < 1.4:
        scores['cv'] = 7
    elif max_cv < 2.7:  # 2.7까지 허용
        scores['cv'] = 4
    else:
        scores['cv'] = 0

    # 3. 스파이크 (10점) - 중간
    a_changes = np.abs(np.diff(a_series))
    b_changes = np.abs(np.diff(b_series))

    if len(a_changes) > 0 and len(b_changes) > 0:
        a_spike = a_changes.max() / (np.median(a_changes) + 1)
        b_spike = b_changes.max() / (np.median(b_changes) + 1)
        max_spike = max(a_spike, b_spike)

        if max_spike < 4.5:
            scores['spike'] = 10
        elif max_spike < 9.5:
            scores['spike'] = 7
        elif max_spike < 17:  # 17까지 허용
            scores['spike'] = 4
        else:
            scores['spike'] = 0
    else:
        scores['spike'] = 5

    # 4. 안정성 (15점) - 중간
    n = len(a_series)

    try:
        mid = n // 2

        if mid >= 12 and mid + lag < len(b_series):
            corr_first = np.corrcoef(
                a_series[:mid],
                b_series[lag:mid+lag]
            )[0, 1]

            corr_second = np.corrcoef(
                a_series[mid:],
                b_series[mid+lag:]
            )[0, 1]

            if not np.isnan(corr_first) and not np.isnan(corr_second):
                consistency = 1 - abs(corr_first - corr_second)

                if consistency > 0.52:  # 중간
                    scores['stability'] = 15
                elif consistency > 0.35:
                    scores['stability'] = 10
                elif consistency > 0.22:
                    scores['stability'] = 6
                else:
                    scores['stability'] = 3
            else:
                scores['stability'] = 5
        else:
            scores['stability'] = 5
    except:
        scores['stability'] = 5

    # 5. 트렌드 일치 (10점) - 중간
    if len(a_series) > 0 and len(b_series) > 0:
        x = np.arange(len(a_series))

        if np.std(a_series) > 0:
            slope_a, _ = np.polyfit(x, a_series, 1)
        else:
            slope_a = 0

        if np.std(b_series) > 0:
            slope_b, _ = np.polyfit(x, b_series, 1)
        else:
            slope_b = 0

        if (slope_a > 0 and slope_b > 0) or (slope_a < 0 and slope_b < 0):
            ratio = abs(slope_a) / (abs(slope_b) + 1e-10)
            if 0.38 < ratio < 2.7:  # 중간
                scores['trend'] = 10
            elif 0.25 < ratio < 4.2:
                scores['trend'] = 6
            else:
                scores['trend'] = 3
        elif slope_a == 0 or slope_b == 0:
            scores['trend'] = 5
        else:
            scores['trend'] = 2
    else:
        scores['trend'] = 4

    # 6. 방향 일치도 (10점) - 중간
    if len(a_changes) > lag and len(b_changes) > lag:
        direction_match = (np.sign(a_changes[:-lag]) == np.sign(b_changes[lag:])).mean()

        if direction_match > 0.64:  # 중간
            scores['direction'] = 10
        elif direction_match > 0.54:
            scores['direction'] = 7
        elif direction_match > 0.44:
            scores['direction'] = 4
        else:
            scores['direction'] = 1
    else:
        scores['direction'] = 4

    # 7. 통계적 유의성 (10점) - 중간
    try:
        if len(a_series) > lag:
            _, p_value = stats.pearsonr(
                a_series[:-lag] if lag > 0 else a_series,
                b_series[lag:] if lag > 0 else b_series
            )

            if p_value < 0.007:  # 중간
                scores['significance'] = 10
            elif p_value < 0.025:
                scores['significance'] = 7
            elif p_value < 0.07:
                scores['significance'] = 4
            else:
                scores['significance'] = 1
        else:
            scores['significance'] = 4
    except:
        scores['significance'] = 4

    # 8. 자기상관 (10점) - 중간
    try:
        if len(b_series) > 1:
            autocorr = np.corrcoef(b_series[:-1], b_series[1:])[0, 1]

            if 0.24 < autocorr < 0.86:  # 중간
                scores['autocorr'] = 10
            elif 0.09 < autocorr < 0.91:
                scores['autocorr'] = 7
            else:
                scores['autocorr'] = 3
        else:
            scores['autocorr'] = 4
    except:
        scores['autocorr'] = 4

    # 9. 이상치 비율 (10점) - 중간
    def count_outliers(series):
        q1 = np.percentile(series, 25)
        q3 = np.percentile(series, 75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()
        return outliers / len(series)

    outlier_ratio_a = count_outliers(a_series)
    outlier_ratio_b = count_outliers(b_series)
    max_outlier = max(outlier_ratio_a, outlier_ratio_b)

    if max_outlier < 0.085:
        scores['outlier'] = 10
    elif max_outlier < 0.16:
        scores['outlier'] = 7
    elif max_outlier < 0.26:  # 26%까지 허용
        scores['outlier'] = 4
    else:
        scores['outlier'] = 1

    # 10. 분포 정규성 (5점) - 중간
    try:
        if len(b_series) >= 8:
            _, p_shapiro = stats.shapiro(b_series)

            if p_shapiro > 0.02:  # 중간
                scores['normality'] = 5
            elif p_shapiro > 0.004:
                scores['normality'] = 3
            else:
                scores['normality'] = 1
        else:
            scores['normality'] = 2
    except:
        scores['normality'] = 2

    total_score = sum(scores.values())

    return total_score, scores


# ============================================================================
# ⭐ 2,700개 목표 노이즈 제거
# ============================================================================

def remove_noise_pairs_for_2700(pairs, pivot, target_count=2700):
    """
    2,700개 목표 노이즈 제거

    원본: 3,368개
    목표: 2,700개 (약 80% 유지)
    전략: 적절한 노이즈 제거 (중간 모드)
    """

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 노이즈 제거 (중간 모드)")
    print("="*80)
    print(f"원본: {len(pairs):,}개")
    print(f"목표: {target_count:,}개 (약 {target_count/len(pairs)*100:.0f}% 유지)")

    quality_scores = []
    detailed_scores = []

    # 품질 점수 계산 (중간 기준)
    print(f"\n품질 평가 중...")
    for row in tqdm(pairs.itertuples(), total=len(pairs), desc="평가"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            quality_scores.append(0)
            detailed_scores.append({})
            continue

        a_series = pivot.loc[leader].values
        b_series = pivot.loc[follower].values

        total_score, scores = calculate_quality_score_moderate(a_series, b_series, lag)

        quality_scores.append(total_score)
        detailed_scores.append(scores)

    # 품질 점수 추가
    pairs_with_score = pairs.copy()
    pairs_with_score['quality_score'] = quality_scores

    # 목표 개수 기반 임계값 계산
    sorted_scores = sorted(quality_scores, reverse=True)

    if len(sorted_scores) >= target_count:
        # 목표 개수번째 점수를 임계값으로
        threshold = sorted_scores[target_count - 1]
        # 5점 단위로 내림
        threshold = max(33, (threshold // 5) * 5)  # 최소 33점
    else:
        threshold = 33  # 최소 임계값

    # 필터링
    pairs_clean = pairs_with_score[
        pairs_with_score['quality_score'] >= threshold
    ].copy()

    # 🔸 여기서는 quality_score를 남겨둬도 되고, 지금처럼 드랍해도 됨
    # (현재 요구사항은 seed 앙상블이므로 그대로 둠)
    pairs_for_model = pairs_clean.drop(columns=['quality_score'], errors='ignore')

    # 통계
    removed = len(pairs) - len(pairs_clean)
    removed_ratio = removed / len(pairs) * 100
    kept_ratio = len(pairs_clean) / len(pairs) * 100

    print(f"\n{'='*80}")
    print(f"✅ 노이즈 제거 완료!")
    print(f"{'='*80}")
    print(f"원본: {len(pairs):,}개")
    print(f"제거: {removed:,}개 ({removed_ratio:.1f}%)")
    print(f"유지: {len(pairs_clean):,}개 ({kept_ratio:.1f}%)")
    print(f"임계값: {threshold}점")

    # 점수 분포
    print(f"\n품질 점수 분포:")
    print(f"  - 평균: {np.mean(quality_scores):.1f}점")
    print(f"  - 중앙값: {np.median(quality_scores):.1f}점")
    print(f"  - 최소: {np.min(quality_scores):.1f}점")
    print(f"  - 최대: {np.max(quality_scores):.1f}점")

    # 선택된 쌍의 점수
    selected_scores = pairs_clean['quality_score'].values
    print(f"\n선택된 쌍의 점수:")
    print(f"  - 평균: {np.mean(selected_scores):.1f}점")
    print(f"  - 최소: {np.min(selected_scores):.1f}점")

    # 구간별 분포
    print(f"\n점수 구간별 분포:")
    bins = [0, 30, 40, 50, 60, 70, 80, 90, 100]
    for i in range(len(bins)-1):
        total_count = sum((np.array(quality_scores) >= bins[i]) & (np.array(quality_scores) < bins[i+1]))
        selected_count = sum((selected_scores >= bins[i]) & (selected_scores < bins[i+1]))
        print(f"  {bins[i]:3d}-{bins[i+1]:3d}점: 전체 {total_count:4,}개, 선택 {selected_count:4,}개")

    # 기준별 평균 점수
    print(f"\n기준별 평균 점수 (선택된 쌍):")
    criteria_names = {
        'zero': '0 비율',
        'cv': '변동계수',
        'spike': '스파이크',
        'stability': '안정성',
        'trend': '트렌드 일치',
        'direction': '방향 일치도',
        'significance': '통계적 유의성',
        'autocorr': '자기상관',
        'outlier': '이상치 비율',
        'normality': '정규성'
    }

    selected_indices = pairs_with_score[pairs_with_score['quality_score'] >= threshold].index
    selected_detailed = [detailed_scores[i] for i in selected_indices if i < len(detailed_scores)]

    for key, name in criteria_names.items():
        scores = [s.get(key, 0) for s in selected_detailed if s]
        if scores:
            if key == 'stability':
                max_score = 15
            elif key == 'normality':
                max_score = 5
            else:
                max_score = 10

            print(f"  {name:15s}: 평균 {np.mean(scores):5.1f}점 (최대 {max_score}점)")

    return pairs_for_model


# ============================================================================
# 조건부 Weight 피처
# ============================================================================

def create_conditional_weight_features(value, weight, value_series):
    """조건부 Weight 피처 4개"""
    value_median = np.median(value_series[value_series > 0]) if (value_series > 0).any() else 0
    value_q75 = np.percentile(value_series[value_series > 0], 75) if (value_series > 0).any() else 0

    if value > value_median:
        weight_for_high_value = weight * 1.5
    else:
        weight_for_high_value = 0

    weight_value_ratio = weight * (value / (value_median + 1))
    exp_weighted = weight * (1 - np.exp(-value / (value_median + 1)))

    if value > value_q75:
        weight_importance = weight * 1.5
    elif value > value_median:
        weight_importance = weight * 1.0
    else:
        weight_importance = weight * 0.1

    return {
        'weight_for_high_value': weight_for_high_value,
        'weight_value_ratio': weight_value_ratio,
        'exp_weighted': exp_weighted,
        'weight_importance': weight_importance
    }


# ============================================================================
# 학습 데이터 구축 (32개 피처)
# ============================================================================

def build_training_data_optimized(pivot, pivot_weight, pairs, months_dt):
    """32개 피처 학습 데이터 생성"""
    months = months_dt
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        for t in range(max(lag, 12), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            b_t_2 = b_series[t - 2]
            b_t_12 = b_series[t - 12]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]
            b_weight_t = b_weight_series[t]
            target_month = months[t + 1].month

            a_t_lag_diff = a_series[t - lag] - a_series[t - lag - 1]
            a_momentum = a_series[t - lag] / (a_series[t - lag - 3] + 1)
            a_roll_mean_3 = (a_series[t-lag] + a_series[t-lag-1] + a_series[t-lag-2]) / 3

            b_diff = b_t - b_t_1
            b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
            b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
            b_roll_mean_3 = (b_series[t] + b_series[t-1] + b_series[t-2]) / 3
            b_roll_mean_6 = (b_series[t-5:t+1]).mean()
            b_roll_std_3 = np.std([b_series[t], b_series[t-1], b_series[t-2]])

            ab_ratio = a_t_lag / (b_t + 1)
            ab_diff = a_t_lag - b_t
            corr_weighted_a = a_t_lag * corr

            b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
            b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
            b_yoy_ratio = b_t / (b_t_12 + 1)
            b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
            recent_12 = b_series[max(0, t-11):t+1]
            b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

            quarter = (target_month - 1) // 3 + 1
            is_year_end = 1 if target_month in [11, 12, 1] else 0
            lag_weight = 1 / (1 + lag)

            b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

            rows.append({
                "b_t": b_t, "b_t_1": b_t_1, "b_t_12": b_t_12, "a_t_lag": a_t_lag,
                "max_corr": corr, "best_lag": float(lag), "month": float(target_month),
                "a_t_lag_diff": a_t_lag_diff, "a_momentum": a_momentum, "a_roll_mean_3": a_roll_mean_3,
                "b_diff": b_diff, "b_pct_change": b_pct_change, "b_yoy_growth": b_yoy_growth,
                "b_roll_mean_3": b_roll_mean_3, "b_roll_mean_6": b_roll_mean_6, "b_roll_std_3": b_roll_std_3,
                "ab_ratio": ab_ratio, "ab_diff": ab_diff, "corr_weighted_a": corr_weighted_a,
                "b_cv": b_cv, "b_acceleration": b_acceleration, "b_yoy_ratio": b_yoy_ratio,
                "b_ma_ratio": b_ma_ratio, "b_percentile_rank": b_percentile_rank,
                "quarter": float(quarter), "is_year_end": is_year_end, "lag_weight": lag_weight,
                "b_weight_for_high_value": b_weight_features['weight_for_high_value'],
                "b_weight_value_ratio": b_weight_features['weight_value_ratio'],
                "b_exp_weighted": b_weight_features['exp_weighted'],
                "b_weight_importance": b_weight_features['weight_importance'],
                "target": np.log1p(b_t_plus_1),
            })

    return pd.DataFrame(rows)


# ============================================================================
# 예측 함수 (seed 앙상블 버전)
# ============================================================================

def predict_optimized(pivot, pivot_weight, pairs, model_lgb_list, model_xgb_list, months_dt):
    """32개 피처 예측 + LGBM/XGB seed 앙상블 + 0.7/0.3 앙상블"""
    months = months_dt
    n_months = len(months)
    t_last = n_months - 1
    t_prev = n_months - 2
    preds = []
    target_month = months[-1].month + 1 if months[-1].month < 12 else 1

    for row in tqdm(pairs.itertuples(index=False), desc="예측"):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue
        if leader not in pivot_weight.index or follower not in pivot_weight.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)
        b_weight_series = pivot_weight.loc[follower].values.astype(float)

        if t_last - lag < 0 or t_last < 12:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        b_t_2 = b_series[t_prev - 1]
        b_t_12 = b_series[t_last - 12]
        a_t_lag = a_series[t_last - lag]
        b_weight_t = b_weight_series[t_last]

        a_t_lag_diff = a_series[t_last - lag] - a_series[t_last - lag - 1]
        a_momentum = a_series[t_last - lag] / (a_series[t_last - lag - 3] + 1)
        a_roll_mean_3 = (a_series[t_last-lag] + a_series[t_last-lag-1] + a_series[t_last-lag-2]) / 3

        b_diff = b_t - b_t_1
        b_pct_change = (b_t - b_t_1) / (b_t_1 + 1)
        b_yoy_growth = (b_t - b_t_12) / (b_t_12 + 1)
        b_roll_mean_3 = (b_series[t_last] + b_series[t_last-1] + b_series[t_last-2]) / 3
        b_roll_mean_6 = b_series[t_last-5:t_last+1].mean()
        b_roll_std_3 = np.std([b_series[t_last], b_series[t_last-1], b_series[t_last-2]])

        ab_ratio = a_t_lag / (b_t + 1)
        ab_diff = a_t_lag - b_t
        corr_weighted_a = a_t_lag * corr

        b_cv = b_roll_std_3 / (b_roll_mean_3 + 1)
        b_acceleration = (b_t - b_t_1) - (b_t_1 - b_t_2)
        b_yoy_ratio = b_t / (b_t_12 + 1)
        b_ma_ratio = b_roll_mean_3 / (b_roll_mean_6 + 1)
        recent_12 = b_series[max(0, t_last-11):t_last+1]
        b_percentile_rank = (b_t >= recent_12).sum() / len(recent_12)

        quarter = (target_month - 1) // 3 + 1
        is_year_end = 1 if target_month in [11, 12, 1] else 0
        lag_weight = 1 / (1 + lag)

        b_weight_features = create_conditional_weight_features(b_t, b_weight_t, b_series)

        X_test = np.array([[
            b_t, b_t_1, b_t_12, a_t_lag, corr, float(lag), float(target_month),
            a_t_lag_diff, a_momentum, a_roll_mean_3,
            b_diff, b_pct_change, b_yoy_growth, b_roll_mean_3, b_roll_mean_6, b_roll_std_3,
            ab_ratio, ab_diff, corr_weighted_a,
            b_cv, b_acceleration, b_yoy_ratio, b_ma_ratio, b_percentile_rank,
            float(quarter), is_year_end, lag_weight,
            b_weight_features['weight_for_high_value'],
            b_weight_features['weight_value_ratio'],
            b_weight_features['exp_weighted'],
            b_weight_features['weight_importance']
        ]])

        # 🔹 여러 seed 모델의 예측을 평균
        pred_lgb_list = [m.predict(X_test)[0] for m in model_lgb_list]
        pred_xgb_list = [m.predict(X_test)[0] for m in model_xgb_list]

        pred_lgb = float(np.mean(pred_lgb_list))
        pred_xgb = float(np.mean(pred_xgb_list))

        # 기존과 동일한 0.7 / 0.3 앙상블
        y_pred_log = 0.7 * pred_lgb + 0.3 * pred_xgb
        y_pred = np.expm1(y_pred_log)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    return pd.DataFrame(preds)


# ============================================================================
# 메인 실행
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("⭐ 2,700개 목표 최적화 버전 (seed 앙상블 포함)")
    print("="*80)

    # ⭐ 2,700개 목표 노이즈 제거
    pairs_for_model = remove_noise_pairs_for_2700(
        pairs=pairs_with_dtw,
        pivot=pivot,
        target_count=2700      # ⭐ 목표: 2,700개
    )

    print(f"\n최종 사용 쌍: {len(pairs_for_model)}개")

    # 학습
    print("\n" + "="*80)
    print("학습 데이터 생성")
    print("="*80)

    df_train_model = build_training_data_optimized(pivot, pivot_weight, pairs_for_model, months_dt)
    print(f"✓ 학습 데이터: {df_train_model.shape}")

    feature_cols = [
        'b_t', 'b_t_1', 'b_t_12', 'a_t_lag', 'max_corr', 'best_lag', 'month',
        'a_t_lag_diff', 'a_momentum', 'a_roll_mean_3',
        'b_diff', 'b_pct_change', 'b_yoy_growth', 'b_roll_mean_3', 'b_roll_mean_6', 'b_roll_std_3',
        'ab_ratio', 'ab_diff', 'corr_weighted_a',
        'b_cv', 'b_acceleration', 'b_yoy_ratio', 'b_ma_ratio', 'b_percentile_rank',
        'quarter', 'is_year_end', 'lag_weight',
        'b_weight_for_high_value', 'b_weight_value_ratio', 'b_exp_weighted', 'b_weight_importance'
    ]

    if not df_train_model.empty:
        print("\n" + "="*80)
        print("모델 학습 (여러 seed 앙상블)")
        print("="*80)

        train_X = df_train_model[feature_cols].values
        train_y = df_train_model["target"].values

        print(f"학습 샘플: {len(train_X):,}, 피처: {len(feature_cols)}")

        # 🔹 여러 seed로 LGBM/XGB 학습
        seeds_lgb = [42, 2024, 999]
        seeds_xgb = [42, 2024, 999]

        lgb_models = []
        xgb_models = []

        print("\nLGBM seed 앙상블 학습 중...")
        for s in seeds_lgb:
            print(f"  - LGBM (seed={s}) 학습 중...")
            model_lgb = LGBMRegressor(
                random_state=s,
                n_estimators=700, learning_rate=0.01, max_depth=8,
                num_leaves=50, min_child_samples=20,
                subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=0.1, min_split_gain=0.01
            )
            model_lgb.fit(train_X, train_y)
            lgb_models.append(model_lgb)
        print("✓ LGBM seed 앙상블 완료")

        print("\nXGBoost seed 앙상블 학습 중...")
        for s in seeds_xgb:
            print(f"  - XGB (seed={s}) 학습 중...")
            model_xgb = XGBRegressor(
                random_state=s,
                n_estimators=900, learning_rate=0.02, max_depth=8,
                subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=0.1,
                tree_method='hist'
            )
            model_xgb.fit(train_X, train_y)
            xgb_models.append(model_xgb)
        print("✓ XGBoost seed 앙상블 완료")

        print("\n" + "="*80)
        print("예측")
        print("="*80)

        submission = predict_optimized(pivot, pivot_weight, pairs_for_model, lgb_models, xgb_models, months_dt)
        print(f"\n✓ 예측 완료: {len(submission)}개")

        date_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        submission.to_csv(f'_result_sy_2700_moderate_seedEns_{date_str}.csv', index=False)

        print("\n✅ 2,700개 목표 버전 (seed 앙상블) 완료!")
        print("  - 원본: 3,368개")
        print("  - 목표: 2,700개")
        print("  - 중간 기준 (적절한 노이즈 제거)")
        print("  - 32개 피처")
        print("  - LGBM + XGB + seed 앙상블 (0.7 / 0.3)")
        print("="*80)


데이터 로드
✓ 데이터 로드 완료: 100개 품목, 43개월

⭐ 2,700개 목표 최적화 버전 (seed 앙상블 포함)

⭐ 2,700개 목표 노이즈 제거 (중간 모드)
원본: 3,368개
목표: 2,700개 (약 80% 유지)

품질 평가 중...


평가: 100%|██████████| 3368/3368 [00:09<00:00, 343.70it/s]



✅ 노이즈 제거 완료!
원본: 3,368개
제거: 611개 (18.1%)
유지: 2,757개 (81.9%)
임계값: 55점

품질 점수 분포:
  - 평균: 65.5점
  - 중앙값: 67.0점
  - 최소: 36.0점
  - 최대: 90.0점

선택된 쌍의 점수:
  - 평균: 69.2점
  - 최소: 55.0점

점수 구간별 분포:
    0- 30점: 전체    0개, 선택    0개
   30- 40점: 전체   29개, 선택    0개
   40- 50점: 전체  205개, 선택    0개
   50- 60점: 전체  708개, 선택  331개
   60- 70점: 전체 1,002개, 선택 1,002개
   70- 80점: 전체 1,212개, 선택 1,212개
   80- 90점: 전체  208개, 선택  208개
   90-100점: 전체    4개, 선택    4개

기준별 평균 점수 (선택된 쌍):
  0 비율           : 평균   9.8점 (최대 10점)
  변동계수           : 평균   7.8점 (최대 10점)
  스파이크           : 평균   6.2점 (최대 10점)
  안정성            : 평균   5.0점 (최대 15점)
  트렌드 일치         : 평균   3.9점 (최대 10점)
  방향 일치도         : 평균  10.0점 (최대 10점)
  통계적 유의성        : 평균   7.7점 (최대 10점)
  자기상관           : 평균   7.8점 (최대 10점)
  이상치 비율         : 평균   8.7점 (최대 10점)
  정규성            : 평균   2.2점 (최대 5점)

최종 사용 쌍: 2757개

학습 데이터 생성
✓ 학습 데이터: (82710, 32)

모델 학습 (여러 seed 앙상블)
학습 샘플: 82,710, 피처: 31

LGBM seed 앙상블 학습 중...
  - LGBM (seed=42) 학습 중...
  - LGBM (seed=20

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
예측: 1924it [01:14, 18.30it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warni


✓ 예측 완료: 2757개

✅ 2,700개 목표 버전 (seed 앙상블) 완료!
  - 원본: 3,368개
  - 목표: 2,700개
  - 중간 기준 (적절한 노이즈 제거)
  - 32개 피처
  - LGBM + XGB + seed 앙상블 (0.7 / 0.3)
